In [ ]:
!pip install -q openai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

FILE: README.md
PATH: /README.md

# RemStroy Assistant — RAG-приложение

Простое RAG-приложение для компании RemStroy (ремонт квартир, установка
оконных и балконных блоков). Отвечает на вопросы на основе собственной
базы знаний, показывает 5 найденных чанков и итоговый AI-ответ.

## Стек
- Python 3.10+
- OpenAI API (chat: gpt-6-luna, embeddings: text-embedding-3-small)
- ChromaDB (PersistentClient, постоянное хранилище на диске)
- Простой HTML/CSS/JS фронтенд

## Структура
    main.py              — HTTP-сервер, точка входа
    config.py            — настройки
    openai_client.py     — обёртка OpenAI
    chroma_store.py      — ChromaDB
    rag_pipeline.py      — RAG-логика (чанкинг, поиск, генерация, память)
    static/              — фронтенд
    knowledge_base/      — демо-документы RemStroy
    chroma_db/           — постоянное хранилище (создаётся автоматически)

## Установка

    pip install -r requirements.txt

## Настройка

Задайте переменную окружения:

    export OPENAI_API_KEY=sk-...

(в Replit / Google AI Studio — через Secrets)

## Запуск

    python main.py

Откроется браузер на http://localhost:8000

## Первый запуск

При первом запуске индекс ChromaDB пуст. Чтобы его заполнить, вызовите
один раз эндпоинт:

    curl -X POST http://localhost:8000/api/reindex

Или нажмите кнопку «Переиндексировать базу» в интерфейсе.

При повторных запусках индекс сохраняется в ./chroma_db/ и не теряется.

FILE: requirements.txt PATH: /requirements.txt

openai>=1.40.0 chromadb>=0.5.0

FILE: config.py
PATH: /config.py

"""Настройки приложения RemStroy Assistant."""

import os

# --- OpenAI ---
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

# Модель генерации (по ТЗ — gpt-6-luna; при необходимости заменить)
CHAT_MODEL = "gpt-6-luna"

# Модель эмбеддингов
EMBEDDING_MODEL = "text-embedding-3-small"

# Параметры генерации
TEMPERATURE = 0.3
MAX_TOKENS = 500

# --- RAG ---
CHUNK_MAX_PARAGRAPHS = 3   # сколько абзацев в одном чанке
CHUNK_OVERLAP = 1          # перекрытие абзацев
TOP_K = 5                  # сколько чанков достаём

# --- Память диалога ---
HISTORY_MAX_MESSAGES = 10  # последние N сообщений (5 обменов)

# --- ChromaDB ---
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "remstroy_knowledge"

# --- База знаний ---
KNOWLEDGE_DIR = "./knowledge_base"

# --- Сервер ---
HOST = "0.0.0.0"
PORT = 8000

FILE: openai_client.py
PATH: /openai_client.py

"""Обёртка над OpenAI API: эмбеддинги и чат."""

from openai import OpenAI
from config import (
    OPENAI_API_KEY, CHAT_MODEL, EMBEDDING_MODEL,
    TEMPERATURE, MAX_TOKENS,
)

_client = OpenAI(api_key=OPENAI_API_KEY)


def get_embedding(text: str) -> list[float]:
    """Возвращает вектор эмбеддинга для строки."""
    response = _client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )
    return response.data[0].embedding


def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    """Батч-эмбеддинги для списка строк."""
    response = _client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]


def chat(messages: list[dict]) -> str:
    """Отправляет сообщения в чат-модель, возвращает текст ответа."""
    response = _client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content

FILE: chroma_store.py
PATH: /chroma_store.py

"""Постоянное векторное хранилище на ChromaDB."""

import chromadb
from chromadb.config import Settings
from config import CHROMA_PATH, COLLECTION_NAME

_client = chromadb.PersistentClient(
    path=CHROMA_PATH,
    settings=Settings(anonymized_telemetry=False),
)

_collection = _client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)


def get_collection():
    return _collection


def reset_collection():
    """Удаляет и пересоздаёт коллекцию."""
    global _collection
    try:
        _client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    _collection = _client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )
    return _collection


def add_chunks(ids, documents, embeddings, metadatas):
    _collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )


def query_chunks(query_embedding, n_results=5):
    return _collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "distances", "metadatas"],
    )


def count() -> int:
    return _collection.count()

FILE: rag_pipeline.py
PATH: /rag_pipeline.py

"""RAG-логика: чанкинг, индексация, поиск, генерация, память."""

import os
import glob

from config import (
    KNOWLEDGE_DIR, CHUNK_MAX_PARAGRAPHS, CHUNK_OVERLAP, TOP_K,
    HISTORY_MAX_MESSAGES,
)
from openai_client import get_embedding, get_embeddings_batch, chat
from chroma_store import (
    add_chunks, query_chunks, reset_collection, count,
)

# Глобальная история диалога (в памяти процесса)
conversation_history: list[dict] = []


# ---------- Чанкинг ----------

def smart_chunk(text: str,
                max_paragraphs: int = CHUNK_MAX_PARAGRAPHS,
                overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Разбивает текст на чанки по абзацам с перекрытием."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: list[str] = []
    i = 0
    while i < len(paragraphs):
        end = min(i + max_paragraphs, len(paragraphs))
        chunk = "\n\n".join(paragraphs[i:end])
        chunks.append(chunk)
        if end >= len(paragraphs):
            break
        i = end - overlap
    return chunks


# ---------- Индексация ----------

def load_knowledge_files() -> list[tuple[str, str]]:
    """Возвращает список (имя_файла, текст)."""
    files = []
    pattern = os.path.join(KNOWLEDGE_DIR, "*.md")
    for path in sorted(glob.glob(pattern)):
        with open(path, "r", encoding="utf-8") as f:
            files.append((os.path.basename(path), f.read()))
    return files


def build_index(verbose: bool = True) -> int:
    """Строит (пересобирает) индекс ChromaDB из knowledge_base/."""
    reset_collection()

    files = load_knowledge_files()
    if not files:
        if verbose:
            print("[RAG] Папка knowledge_base пуста.")
        return 0

    ids: list[str] = []
    documents: list[str] = []
    metadatas: list[dict] = []
    embeddings: list[list[float]] = []

    for filename, text in files:
        chunks = smart_chunk(text)
        for idx, chunk in enumerate(chunks):
            ids.append(f"{filename}::{idx}")
            documents.append(chunk)
            metadatas.append({"source": filename, "chunk": idx})

    # Батч-эмбеддинги (можно разбить на пакеты, если чанков много)
    BATCH = 64
    for start in range(0, len(documents), BATCH):
        batch = documents[start:start + BATCH]
        embeddings.extend(get_embeddings_batch(batch))

    add_chunks(ids, documents, embeddings, metadatas)

    if verbose:
        print(f"[RAG] Проиндексировано чанков: {len(ids)}")
    return len(ids)


def ensure_index():
    """Если индекс пуст — строит его."""
    if count() == 0:
        build_index()


# ---------- Поиск ----------

def search(query: str, n_results: int = TOP_K) -> list[dict]:
    """Возвращает список словарей: text, source, score."""
    query_emb = get_embedding(query)
    results = query_chunks(query_emb, n_results=n_results)

    docs = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]
    metas = results.get("metadatas", [[]])[0]

    hits = []
    for doc, dist, meta in zip(docs, dists, metas):
        # cosine distance -> similarity
        score = 1.0 - float(dist)
        hits.append({
            "text": doc,
            "source": meta.get("source", "unknown"),
            "chunk": meta.get("chunk", -1),
            "score": round(score, 4),
        })
    return hits


# ---------- Генерация ----------

SYSTEM_PROMPT = (
    "Ты — ассистент компании RemStroy. Компания занимается ремонтом "
    "жилых квартир и установкой оконных и балконных блоков "
    "(2- и 3-слойных) по различным технологиям. "
    "Отвечай на русском языке, опираясь на предоставленный контекст. "
    "Если в контексте нет информации — честно скажи об этом."
)


def generate_answer(question: str, hits: list[dict]) -> str:
    context_parts = []
    for i, h in enumerate(hits, 1):
        context_parts.append(
            f"[Фрагмент {i} | источник: {h['source']}]\n{h['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(conversation_history[-HISTORY_MAX_MESSAGES:])
    messages.append({
        "role": "user",
        "content": f"Контекст:\n{context}\n\nВопрос: {question}",
    })

    return chat(messages)


# ---------- Основной вход ----------

def rag_query(question: str) -> dict:
    ensure_index()

    hits = search(question, n_results=TOP_K)
    answer = generate_answer(question, hits)

    conversation_history.append({"role": "user", "content": question})
    conversation_history.append({"role": "assistant", "content": answer})
    if len(conversation_history) > HISTORY_MAX_MESSAGES:
        conversation_history[:] = conversation_history[-HISTORY_MAX_MESSAGES:]

    return {"chunks": hits, "answer": answer}


def reset_history():
    conversation_history.clear()

FILE: main.py
PATH: /main.py

"""HTTP-сервер RemStroy Assistant."""

import json
import os
import http.server
import socketserver
import webbrowser
from threading import Timer

from config import HOST, PORT
from rag_pipeline import rag_query, build_index, reset_history
from chroma_store import count

STATIC_DIR = os.path.join(os.path.dirname(__file__), "static")


class Handler(http.server.BaseHTTPRequestHandler):

    # ---------- helpers ----------
    def _send_json(self, payload: dict, status: int = 200):
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _read_json(self) -> dict:
        length = int(self.headers.get("Content-Length", 0))
        if length == 0:
            return {}
        raw = self.rfile.read(length)
        try:
            return json.loads(raw.decode("utf-8"))
        except Exception:
            return {}

    def _serve_static(self, rel_path: str):
        safe_rel = rel_path.lstrip("/").replace("..", "")
        full = os.path.join(STATIC_DIR, safe_rel)
        if not os.path.isfile(full):
            self.send_error(404, "Not found")
            return
        ext = os.path.splitext(full)[1].lower()
        content_type = {
            ".html": "text/html; charset=utf-8",
            ".css":  "text/css; charset=utf-8",
            ".js":   "application/javascript; charset=utf-8",
            ".svg":  "image/svg+xml",
            ".png":  "image/png",
        }.get(ext, "application/octet-stream")
        with open(full, "rb") as f:
            data = f.read()
        self.send_response(200)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    # ---------- routing ----------
    def do_GET(self):
        if self.path in ("/", "/index.html"):
            self._serve_static("index.html")
        elif self.path == "/api/status":
            self._send_json({"indexed_chunks": count()})
        else:
            self._serve_static(self.path)

    def do_POST(self):
        if self.path == "/api/query":
            data = self._read_json()
            question = (data.get("question") or "").strip()
            if not question:
                self._send_json({"error": "Пустой вопрос"}, status=400)
                return
            try:
                result = rag_query(question)
                self._send_json(result)
            except Exception as e:
                self._send_json({"error": str(e)}, status=500)

        elif self.path == "/api/reindex":
            try:
                n = build_index()
                reset_history()
                self._send_json({"indexed_chunks": n})
            except Exception as e:
                self._send_json({"error": str(e)}, status=500)

        elif self.path == "/api/reset":
            reset_history()
            self._send_json({"ok": True})

        else:
            self.send_error(404, "Not found")

    def log_message(self, fmt, *args):
        # Тише в консоли
        return


def run():
    with socketserver.TCPServer((HOST, PORT), Handler) as httpd:
        url = f"http://localhost:{PORT}"
        print(f"[RemStroy] Сервер запущен: {url}")
        Timer(1.0, lambda: webbrowser.open(url)).start()
        try:
            httpd.serve_forever()
        except KeyboardInterrupt:
            print("\n[RemStroy] Остановлено.")


if __name__ == "__main__":
    run()

FILE: static/index.html
PATH: /static/index.html

<!DOCTYPE html>
<html lang="ru">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>RemStroy Assistant</title>
  <link rel="stylesheet" href="/style.css">
</head>
<body>

  <header class="header">
    <div class="logo">
      <svg width="32" height="32" viewBox="0 0 32 32" aria-hidden="true">
        <rect x="2" y="2" width="28" height="28" rx="6"
              fill="#2f3542"/>
        <path d="M8 22 L8 12 L13 8 L18 12 L18 22"
              stroke="#ffffff" stroke-width="2"
              fill="none" stroke-linejoin="round"/>
        <rect x="14" y="16" width="3" height="6" fill="#ffffff"/>
      </svg>
      <div class="logo-text">
        <span class="logo-title">RemStroy</span>
        <span class="logo-sub">Ассистент по ремонту и окнам</span>
      </div>
    </div>
    <div class="header-actions">
      <button id="btn-reindex" class="btn-ghost" title="Пересобрать индекс">
        Переиндексировать
      </button>
      <button id="btn-reset" class="btn-ghost" title="Сбросить диалог">
        Новый диалог
      </button>
      <span id="status" class="status"></span>
    </div>
  </header>

  <main class="container">

    <section class="chat" id="chat">
      <div class="message assistant">
        <div class="bubble">
          Здравствуйте! Я ассистент RemStroy. Задайте вопрос о ремонте
          квартир, установке оконных или балконных блоков.
        </div>
      </div>
    </section>

    <section class="composer">
      <textarea id="question" rows="2"
        placeholder="Например: Сколько стоит установка трёхслойного окна?"></textarea>
      <button id="btn-send" class="btn-primary">Спросить</button>
    </section>

  </main>

  <script src="/app.js"></script>
</body>
</html>

FILE: static/style.css
PATH: /static/style.css

/* Нейтральный минималистичный стиль */

* { box-sizing: border-box; }

html, body {
  margin: 0;
  padding: 0;
  background: #f5f6f8;
  color: #2f3542;
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
               "Helvetica Neue", Arial, sans-serif;
  font-size: 15px;
  line-height: 1.5;
}

/* ---------- header ---------- */

.header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  padding: 14px 24px;
  background: #ffffff;
  border-bottom: 1px solid #e3e5e8;
}

.logo { display: flex; align-items: center; gap: 12px; }

.logo-text { display: flex; flex-direction: column; }

.logo-title {
  font-weight: 600;
  font-size: 16px;
  color: #2f3542;
}

.logo-sub {
  font-size: 12px;
  color: #8a8f98;
}

.header-actions { display: flex; align-items: center; gap: 10px; }

.status {
  font-size: 12px;
  color: #8a8f98;
  margin-left: 8px;
}

/* ---------- buttons ---------- */

.btn-primary, .btn-ghost {
  border-radius: 8px;
  padding: 8px 14px;
  font-size: 14px;
  cursor: pointer;
  border: 1px solid transparent;
  transition: background .15s ease, border-color .15s ease;
}

.btn-primary {
  background: #2f3542;
  color: #ffffff;
}
.btn-primary:hover { background: #1f242e; }
.btn-primary:disabled {
  background: #a4a9b3; cursor: not-allowed;
}

.btn-ghost {
  background: #ffffff;
  color: #2f3542;
  border-color: #d5d8dd;
}
.btn-ghost:hover { background: #f0f1f3; }

/* ---------- layout ---------- */

.container {
  max-width: 880px;
  margin: 0 auto;
  padding: 20px 24px 40px;
}

/* ---------- chat ---------- */

.chat {
  display: flex;
  flex-direction: column;
  gap: 16px;
  margin-bottom: 16px;
}

.message { display: flex; }
.message.user { justify-content: flex-end; }
.message.assistant { justify-content: flex-start; }

.bubble {
  max-width: 100%;
  padding: 12px 16px;
  border-radius: 12px;
  background: #ffffff;
  border: 1px solid #e3e5e8;
  white-space: pre-wrap;
  word-wrap: break-word;
}

.message.user .bubble {
  background: #eef0f3;
  border-color: #e0e3e7;
}

/* ---------- chunks ---------- */

.chunks {
  margin-top: 12px;
  border-top: 1px dashed #dcdfe4;
  padding-top: 10px;
}

.chunks-title {
  font-size: 13px;
  font-weight: 600;
  color: #6b7280;
  margin-bottom: 8px;
  cursor: pointer;
  user-select: none;
}

.chunk {
  background: #fafbfc;
  border: 1px solid #e8eaee;
  border-radius: 8px;
  padding: 10px 12px;
  margin-bottom: 8px;
  font-size: 13px;
}

.chunk-head {
  display: flex;
  justify-content: space-between;
  gap: 8px;
  font-size: 12px;
  color: #8a8f98;
  margin-bottom: 6px;
}

.chunk-score { font-variant-numeric: tabular-nums; }

.chunk-text {
  color: #3b414d;
  white-space: pre-wrap;
}

/* ---------- composer ---------- */

.composer {
  display: flex;
  gap: 10px;
  align-items: flex-end;
  position: sticky;
  bottom: 16px;
  background: #f5f6f8;
  padding-top: 8px;
}

#question {
  flex: 1;
  resize: vertical;
  min-height: 48px;
  max-height: 160px;
  padding: 12px 14px;
  border-radius: 10px;
  border: 1px solid #d5d8dd;
  background: #ffffff;
  font: inherit;
  color: #2f3542;
  outline: none;
}
#question:focus { border-color: #a4a9b3; }

/* ---------- misc ---------- */

.error {
  color: #b00020;
  background: #fdecef;
  border: 1px solid #f5c6cf;
  padding: 10px 12px;
  border-radius: 8px;
}

.loading {
  font-size: 13px;
  color: #8a8f98;
}

FILE: static/app.js
PATH: /static/app.js

"use strict";

const chatEl = document.getElementById("chat");
const questionEl = document.getElementById("question");
const sendBtn = document.getElementById("btn-send");
const reindexBtn = document.getElementById("btn-reindex");
const resetBtn = document.getElementById("btn-reset");
const statusEl = document.getElementById("status");

// ---------- утилиты ----------

function escapeHtml(s) {
  return String(s)
    .replace(/&/g, "&amp;")
    .replace(/</g, "&lt;")
    .replace(/>/g, "&gt;");
}

function appendUserMessage(text) {
  const wrap = document.createElement("div");
  wrap.className = "message user";
  wrap.innerHTML = `<div class="bubble">${escapeHtml(text)}</div>`;
  chatEl.appendChild(wrap);
  scrollToBottom();
}

function appendAssistantMessage(answer, chunks) {
  const wrap = document.createElement("div");
  wrap.className = "message assistant";

  const bubble = document.createElement("div");
  bubble.className = "bubble";

  const answerDiv = document.createElement("div");
  answerDiv.textContent = answer;
  bubble.appendChild(answerDiv);

  if (Array.isArray(chunks) && chunks.length > 0) {
    const details = document.createElement("div");
    details.className = "chunks";

    const title = document.createElement("div");
    title.className = "chunks-title";
    title.textContent = `Найденные фрагменты (${chunks.length}) — показать`;
    details.appendChild(title);

    const list = document.createElement("div");
    list.style.display = "none";

    chunks.forEach((c, i) => {
      const item = document.createElement("div");
      item.className = "chunk";

      const head = document.createElement("div");
      head.className = "chunk-head";
      head.innerHTML =
        `<span>#${i + 1} · ${escapeHtml(c.source || "unknown")}` +
        ` · чанк ${c.chunk ?? "-"}</span>` +
        `<span class="chunk-score">релевантность: ${c.score}</span>`;
      item.appendChild(head);

      const txt = document.createElement("div");
      txt.className = "chunk-text";
      txt.textContent = c.text;
      item.appendChild(txt);

      list.appendChild(item);
    });

    title.addEventListener("click", () => {
      const open = list.style.display !== "none";
      list.style.display = open ? "none" : "block";
      title.textContent =
        `Найденные фрагменты (${chunks.length}) — ` +
        (open ? "показать" : "скрыть");
    });

    details.appendChild(list);
    bubble.appendChild(details);
  }

  wrap.appendChild(bubble);
  chatEl.appendChild(wrap);
  scrollToBottom();
}

function appendError(text) {
  const wrap = document.createElement("div");
  wrap.className = "message assistant";
  wrap.innerHTML = `<div class="bubble error">${escapeHtml(text)}</div>`;
  chatEl.appendChild(wrap);
  scrollToBottom();
}

function appendLoading() {
  const wrap = document.createElement("div");
  wrap.className = "message assistant";
  wrap.id = "loading-bubble";
  wrap.innerHTML = `<div class="bubble loading">Ищу ответ…</div>`;
  chatEl.appendChild(wrap);
  scrollToBottom();
  return wrap;
}

function scrollToBottom() {
  window.scrollTo({ top: document.body.scrollHeight, behavior: "smooth" });
}

function setBusy(isBusy) {
  sendBtn.disabled = isBusy;
  sendBtn.textContent = isBusy ? "Отправка…" : "Спросить";
}

// ---------- статус ----------

async function refreshStatus() {
  try {
    const r = await fetch("/api/status");
    const data = await r.json();
    statusEl.textContent = `чанков в индексе: ${data.indexed_chunks}`;
  } catch (e) {
    statusEl.textContent = "";
  }
}

// ---------- основные действия ----------

async function ask() {
  const question = questionEl.value.trim();
  if (!question) return;

  appendUserMessage(question);
  questionEl.value = "";
  setBusy(true);

  const loading = appendLoading();

  try {
    const resp = await fetch("/api/query", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ question }),
    });
    const data = await resp.json();

    loading.remove();

    if (data.error) {
      appendError("Ошибка: " + data.error);
    } else {
      appendAssistantMessage(data.answer || "", data.chunks || []);
    }
  } catch (e) {
    loading.remove();
    appendError("Не удалось получить ответ: " + e.message);
  } finally {
    setBusy(false);
    questionEl.focus();
  }
}

async function reindex() {
  reindexBtn.disabled = true;
  reindexBtn.textContent = "Индексация…";
  try {
    const r = await fetch("/api/reindex", { method: "POST" });
    const data = await r.json();
    if (data.error) {
      appendError("Ошибка индексации: " + data.error);
    } else {
      appendAssistantMessage(
        `База знаний проиндексирована. Чанков: ${data.indexed_chunks}.`,
        []
      );
      refreshStatus();
    }
  } catch (e) {
    appendError("Не удалось переиндексировать: " + e.message);
  } finally {
    reindexBtn.disabled = false;
    reindexBtn.textContent = "Переиндексировать";
  }
}

async function resetChat() {
  try {
    await fetch("/api/reset", { method: "POST" });
  } catch (e) { /* ignore */ }
  chatEl.innerHTML = "";
  appendAssistantMessage(
    "Диалог сброшен. Задайте новый вопрос.",
    []
  );
}

// ---------- события ----------

sendBtn.addEventListener("click", ask);

questionEl.addEventListener("keydown", (e) => {
  if (e.key === "Enter" && !e.shiftKey) {
    e.preventDefault();
    ask();
  }
});

reindexBtn.addEventListener("click", reindex);
resetBtn.addEventListener("click", resetChat);

refreshStatus();

FILE: knowledge_base/remont_services.md
PATH: /knowledge_base/remont_services.md

# Трёхслойные ПВХ-окна

Трёхслойные ПВХ-окна — это конструкция с трёхкамерным стеклопакетом
(четыре стекла, три воздушные камеры). Такие окна дают максимальную
тепло- и шумоизоляцию и рекомендованы для холодных регионов,
шумных магистралей и панорамного остекления.

## Характеристики

Ширина профиля — 70–90 мм. Количество камер профиля — 5–7.
Сопротивление теплопередаче — 0,58–0,72 м²·°C/Вт.
Шумоизоляция — до 42 дБ. Светопропускание — около 72–76%.

Трёхслойные окна снижают теплопотери на 25–30% по сравнению
с двухслойными. Это уменьшает расходы на отопление и защищает
от уличного шума.

## Технологии

Используются профильные системы Rehau Geneo, Veka Softline,
KBE Expert. Стеклопакеты — двухкамерные энергосберегающие с
аргоновым заполнением и тёплой дистанционной рамкой. Возможно
нанесение мультифункционального покрытия, защищающего от
солнечного тепла летом.

## Цены

Стоимость трёхслойного окна «под ключ» — от 18 000 ₽ за м².
Итоговая цена зависит от площади, профиля, стеклопакета и
фурнитуры. При заказе от 5 окон — скидка 7%. При заказе от
10 окон — скидка 12%.

FILE: knowledge_base/balkonnye_bloki.md
PATH: /knowledge_base/balkonnye_bloki.md

# Балконные блоки и остекление

RemStroy устанавливает балконные блоки (окно + балконная дверь)
и выполняет остекление балконов и лоджий «под ключ».

## Балконные блоки

Балконный блок состоит из оконной части и балконной двери,
объединённых в единую конструкцию. Мы изготавливаем блоки
на двухслойных и трёхслойных ПВХ-профилях. Ширина двери —
от 60 до 90 см, высота — до 230 см. Возможны варианты
с антресольной фрамугой и поворотно-откидной створкой.

## Остекление балконов

Холодное остекление — алюминиевый профиль без утепления.
Подходит для защиты от осадков и пыли. Цена — от 8 000 ₽ за м².

Тёплое остекление — ПВХ-профиль с двухкамерным или
трёхкамерным стеклопакетом, с утеплением парапета и пола.
Позволяет присоединить балкон к комнате. Цена — от 15 000 ₽ за м².

## Утепление и отделка

При тёплом остеклении мы утепляем парапет, пол и потолок
минеральной ватой или пенополистиролом, монтируем пароизоляцию
и выполняем чистовую отделку: пластиковые панели, ламинат,
декоративная штукатурка. Срок работ — от 5 до 10 дней.

FILE: knowledge_base/tehnologii_montazha.md
PATH: /knowledge_base/tehnologii_montazha.md

# Технологии монтажа окон и балконных блоков

RemStroy выполняет монтаж по ГОСТ 30971-2012 и по технологии
«тёплого монтажа». Тип монтажа влияет на тепло- и шумоизоляцию
и на срок службы конструкции.

## Стандартный монтаж по ГОСТ

Демонтаж старого блока, очистка проёма, установка нового блока
на анкерные пластины или дюбели. Швы заполняются монтажной
пеной, снаружи — паропроницаемая лента (ПСУЛ), внутри —
пароизоляционная лента. Такой монтаж подходит для большинства
жилых помещений.

## Тёплый монтаж

Технология с использованием пароизоляционных и
паропроницаемых лент по всему периметру проёма. Швы
заполняются двухкомпонентной пеной. Внутренний и наружный
контуры герметизируются отдельно. Тёплый монтаж даёт
максимальную защиту от продувания и промерзания,
рекомендован для трёхслойных окон и панорамного остекления.

## Сроки и гарантия на монтаж

Стандартный монтаж одного окна — 2–3 часа. Балконный блок —
4–6 часов. Тёплый монтаж — на 30–50% дольше. Гарантия на
монтажные работы — 3 года, на конструкцию — 5 лет.


FILE: knowledge_base/garantii_i_sroki.md
PATH: /knowledge_base/garantii_i_sroki.md

# Гарантии и сроки RemStroy

## Гарантия на окна и балконные блоки

На профиль и стеклопакет — 5 лет. На фурнитуру — 3 года.
На монтажные работы — 3 года. Гарантия действует при
соблюдении правил эксплуатации: регулярная смазка фурнитуры,
своевременная регулировка створок, отсутствие механических
повреждений.

## Гарантия на ремонт квартир

На общестроительные работы — 2 года. На инженерные системы
(электрика, сантехника) — 3 года. На чистовую отделку — 1 год.
Гарантийные обязательства фиксируются в договоре и акте
приёмки.

## Сроки выполнения

Замер — в течение 1–2 рабочих дней после заявки. Изготовление
окон — 5–10 рабочих дней. Монтаж — 1 день. Ремонт квартиры —
от 2 недель (косметический) до 8 месяцев (дизайнерский).

## Рекламации

При обнаружении дефекта в течение гарантийного срока
оставьте заявку по телефону или через сайт. Выезд мастера —
в течение 3 рабочих дней. Устранение дефекта — в течение
14 дней с момента подтверждения.

FILE: knowledge_base/ceny_i_usloviya.md
PATH: /knowledge_base/ceny_i_usloviya.md

# Цены и условия работы RemStroy

## Окна и балконные блоки

Двухслойные ПВХ-окна — от 12 000 ₽ за м² «под ключ».
Трёхслойные ПВХ-окна — от 18 000 ₽ за м² «под ключ».
Балконный блок (окно + дверь) — от 25 000 ₽.
Холодное остекление балкона — от 8 000 ₽ за м².
Тёплое остекление балкона — от 15 000 ₽ за м².

## Ремонт квартир

Косметический ремонт — от 4 500 ₽ за м².
Капитальный ремонт — от 9 000 ₽ за м².
Дизайнерский ремонт — от 15 000 ₽ за м².
Цены указаны за работу без стоимости чистовых материалов.

## Скидки и рассрочка

При заказе от 5 окон — скидка 7%. От 10 окон — скидка 12%.
Пенсионерам — скидка 5% на монтаж. При комплексном заказе
(ремонт + окна) — скидка 10% на окна.

Рассрочка на 6 или 12 месяцев без процентов при заказе
от 100 000 ₽. Оформление — через банк-партнёр, требуется
паспорт РФ.

## Оплата

Предоплата 30% после подписания договора. 40% — после
доставки материалов. 30% — после приёмки работ. Безналичный
расчёт для юридических лиц, наличные и карта — для физических.

In [ ]:
!pip install -q openai chromadb

In [ ]:
!pip show openai chromadb

Name: openai
Version: 2.54.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: 
---
Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


In [ ]:
import os

os.makedirs("/content/knowledge_base", exist_ok=True)
os.makedirs("/content/static", exist_ok=True)

# ========== config.py ==========
with open("/content/config.py", "w", encoding="utf-8") as f:
    f.write('''import os

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
CHAT_MODEL = "gemini-3.6-flash"
EMBEDDING_MODEL = "gemini-embedding-001"

TEMPERATURE = 0.3
MAX_TOKENS = 500

CHUNK_MAX_PARAGRAPHS = 3
CHUNK_OVERLAP = 1
TOP_K = 5

HISTORY_MAX_MESSAGES = 10

CHROMA_PATH = "/content/chroma_db"
COLLECTION_NAME = "remstroy_knowledge"

KNOWLEDGE_DIR = "/content/knowledge_base"

HOST = "0.0.0.0"
PORT = 8000
''')

# ========== openai_client.py ==========
with open("/content/openai_client.py", "w", encoding="utf-8") as f:
    f.write('''from openai import OpenAI
from config import (
    GEMINI_API_KEY, BASE_URL, CHAT_MODEL, EMBEDDING_MODEL,
    TEMPERATURE, MAX_TOKENS,
)

_client = OpenAI(api_key=GEMINI_API_KEY, base_url=BASE_URL)


def get_embedding(text):
    response = _client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


def get_embeddings_batch(texts):
    response = _client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


def chat(messages):
    response = _client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content
''')

# ========== chroma_store.py ==========
with open("/content/chroma_store.py", "w", encoding="utf-8") as f:
    f.write('''import chromadb
from config import CHROMA_PATH, COLLECTION_NAME

_client = chromadb.PersistentClient(path=CHROMA_PATH)
_collection = _client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)


def get_collection():
    return _collection


def reset_collection():
    global _collection
    try:
        _client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    _collection = _client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )
    return _collection


def add_chunks(ids, documents, embeddings, metadatas):
    _collection.add(
        ids=ids, documents=documents,
        embeddings=embeddings, metadatas=metadatas,
    )


def query_chunks(query_embedding, n_results=5):
    return _collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "distances", "metadatas"],
    )


def count():
    return _collection.count()
''')

print("✓ config.py, openai_client.py, chroma_store.py созданы")
print("Файлы:", sorted(os.listdir("/content")))

✓ config.py, openai_client.py, chroma_store.py созданы
Файлы: ['.config', 'chroma_store.py', 'config.py', 'knowledge_base', 'openai_client.py', 'sample_data', 'static']


In [ ]:
# ========== rag_pipeline.py ==========
with open("/content/rag_pipeline.py", "w", encoding="utf-8") as f:
    f.write('''import os
import glob

from config import (
    KNOWLEDGE_DIR, CHUNK_MAX_PARAGRAPHS, CHUNK_OVERLAP, TOP_K,
    HISTORY_MAX_MESSAGES,
)
from openai_client import get_embedding, get_embeddings_batch, chat
from chroma_store import add_chunks, query_chunks, reset_collection, count

conversation_history = []


def smart_chunk(text, max_paragraphs=CHUNK_MAX_PARAGRAPHS, overlap=CHUNK_OVERLAP):
    paragraphs = [p.strip() for p in text.split("\\n\\n") if p.strip()]
    chunks = []
    i = 0
    while i < len(paragraphs):
        end = min(i + max_paragraphs, len(paragraphs))
        chunk = "\\n\\n".join(paragraphs[i:end])
        chunks.append(chunk)
        if end >= len(paragraphs):
            break
        i = end - overlap
    return chunks


def load_knowledge_files():
    files = []
    pattern = os.path.join(KNOWLEDGE_DIR, "*.md")
    for path in sorted(glob.glob(pattern)):
        with open(path, "r", encoding="utf-8") as fh:
            files.append((os.path.basename(path), fh.read()))
    return files


def build_index(verbose=True):
    reset_collection()
    files = load_knowledge_files()
    if not files:
        if verbose:
            print("[RAG] Папка knowledge_base пуста.")
        return 0

    ids, documents, metadatas, embeddings = [], [], [], []
    for filename, text in files:
        chunks = smart_chunk(text)
        for idx, chunk in enumerate(chunks):
            ids.append(f"{filename}::{idx}")
            documents.append(chunk)
            metadatas.append({"source": filename, "chunk": idx})

    BATCH = 64
    for start in range(0, len(documents), BATCH):
        batch = documents[start:start + BATCH]
        embeddings.extend(get_embeddings_batch(batch))

    add_chunks(ids, documents, embeddings, metadatas)
    if verbose:
        print(f"[RAG] Проиндексировано чанков: {len(ids)}")
    return len(ids)


def ensure_index():
    if count() == 0:
        build_index()


def search(query, n_results=TOP_K):
    query_emb = get_embedding(query)
    results = query_chunks(query_emb, n_results=n_results)
    docs = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    hits = []
    for doc, dist, meta in zip(docs, dists, metas):
        score = 1.0 - float(dist)
        hits.append({
            "text": doc,
            "source": meta.get("source", "unknown"),
            "chunk": meta.get("chunk", -1),
            "score": round(score, 4),
        })
    return hits


SYSTEM_PROMPT = (
    "Ты — ассистент компании RemStroy. Компания занимается ремонтом "
    "жилых квартир и установкой оконных и балконных блоков "
    "(2- и 3-слойных) по различным технологиям. "
    "Отвечай на русском языке, опираясь на предоставленный контекст. "
    "Если в контексте нет информации — честно скажи об этом."
)


def generate_answer(question, hits):
    context_parts = []
    for i, h in enumerate(hits, 1):
        context_parts.append(
            f"[Фрагмент {i} | источник: {h['source']}]\\n{h['text']}"
        )
    context = "\\n\\n---\\n\\n".join(context_parts)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(conversation_history[-HISTORY_MAX_MESSAGES:])
    messages.append({
        "role": "user",
        "content": f"Контекст:\\n{context}\\n\\nВопрос: {question}",
    })
    return chat(messages)


def rag_query(question):
    ensure_index()
    hits = search(question, n_results=TOP_K)
    answer = generate_answer(question, hits)
    conversation_history.append({"role": "user", "content": question})
    conversation_history.append({"role": "assistant", "content": answer})
    if len(conversation_history) > HISTORY_MAX_MESSAGES:
        conversation_history[:] = conversation_history[-HISTORY_MAX_MESSAGES:]
    return {"chunks": hits, "answer": answer}


def reset_history():
    conversation_history.clear()
''')

# ========== main.py ==========
with open("/content/main.py", "w", encoding="utf-8") as f:
    f.write('''import json
import os
import http.server
import socketserver

from config import HOST, PORT
from rag_pipeline import rag_query, build_index, reset_history
from chroma_store import count

STATIC_DIR = "/content/static"


class Handler(http.server.BaseHTTPRequestHandler):
    def _send_json(self, payload, status=200):
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _read_json(self):
        length = int(self.headers.get("Content-Length", 0))
        if length == 0:
            return {}
        raw = self.rfile.read(length)
        try:
            return json.loads(raw.decode("utf-8"))
        except Exception:
            return {}

    def _serve_static(self, rel_path):
        safe_rel = rel_path.lstrip("/").replace("..", "")
        full = os.path.join(STATIC_DIR, safe_rel)
        if not os.path.isfile(full):
            self.send_error(404, "Not found")
            return
        ext = os.path.splitext(full)[1].lower()
        content_type = {
            ".html": "text/html; charset=utf-8",
            ".css":  "text/css; charset=utf-8",
            ".js":   "application/javascript; charset=utf-8",
            ".svg":  "image/svg+xml",
        }.get(ext, "application/octet-stream")
        with open(full, "rb") as fh:
            data = fh.read()
        self.send_response(200)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if self.path in ("/", "/index.html"):
            self._serve_static("index.html")
        elif self.path == "/api/status":
            self._send_json({"indexed_chunks": count()})
        else:
            self._serve_static(self.path)

    def do_POST(self):
        if self.path == "/api/query":
            data = self._read_json()
            question = (data.get("question") or "").strip()
            if not question:
                self._send_json({"error": "Пустой вопрос"}, status=400)
                return
            try:
                result = rag_query(question)
                self._send_json(result)
            except Exception as e:
                self._send_json({"error": str(e)}, status=500)
        elif self.path == "/api/reindex":
            try:
                n = build_index()
                reset_history()
                self._send_json({"indexed_chunks": n})
            except Exception as e:
                self._send_json({"error": str(e)}, status=500)
        elif self.path == "/api/reset":
            reset_history()
            self._send_json({"ok": True})
        else:
            self.send_error(404, "Not found")

    def log_message(self, fmt, *args):
        return


def run():
    with socketserver.TCPServer((HOST, PORT), Handler) as httpd:
        print(f"[RemStroy] Сервер запущен на порту {PORT}")
        httpd.serve_forever()


if __name__ == "__main__":
    run()
''')

print("✓ rag_pipeline.py и main.py созданы")
print("Файлы:", sorted(os.listdir("/content")))

✓ rag_pipeline.py и main.py созданы
Файлы: ['.config', 'chroma_store.py', 'config.py', 'knowledge_base', 'main.py', 'openai_client.py', 'rag_pipeline.py', 'sample_data', 'static']


In [ ]:
KB = "/content/knowledge_base"

# ========== remont_services.md ==========
with open(f"{KB}/remont_services.md", "w", encoding="utf-8") as f:
    f.write('''# Ремонт квартир — услуги RemStroy

Компания RemStroy выполняет ремонт жилых квартир любой сложности. Мы работаем с новостройками и вторичным жильём, с полной или частичной отделкой. Все работы выполняются по договору с фиксированной сметой.

## Виды ремонта

Косметический ремонт — обновление отделки без вмешательства в инженерные системы и планировку. Включает покраску стен, поклейку обоев, замену напольного покрытия, мелкий ремонт потолков. Срок выполнения — от 2 до 4 недель в зависимости от площади квартиры.

Капитальный ремонт — полная замена отделки, инженерных коммуникаций (электрика, сантехника), выравнивание стен и полов, замена окон и дверей. Срок выполнения — от 2 до 4 месяцев.

Дизайнерский ремонт — ремонт по индивидуальному проекту дизайнера, с перепланировкой, авторским надзором и использованием премиальных материалов. Срок выполнения — от 4 до 8 месяцев.

## Этапы работ

Каждый ремонт проходит через фиксированные этапы: замер и проектирование, демонтаж, черновые работы, монтаж инженерных систем, чистовая отделка, финальная приёмка. На каждом этапе заказчик получает фотоотчёт и может внести правки до начала следующего этапа.

## Что входит в стоимость

В стоимость ремонта входят работы, черновые материалы и вывоз строительного мусора. Чистовые материалы (обои, плитка, краска, напольные покрытия) заказчик выбирает и оплачивает отдельно либо заказывает через нас с наценкой 10%.
''')

# ========== okna_pvh_2sloy.md ==========
with open(f"{KB}/okna_pvh_2sloy.md", "w", encoding="utf-8") as f:
    f.write('''# Двухслойные ПВХ-окна

Двухслойные ПВХ-окна — это конструкция с двухкамерным стеклопакетом (три стекла, две воздушные камеры). Такие окна обеспечивают базовую тепло- и шумоизоляцию и подходят для большинства жилых помещений в средней полосе.

## Характеристики

Ширина профиля — 58–70 мм. Количество камер профиля — 3–5. Сопротивление теплопередаче стеклопакета — 0,47–0,52 м²·°C/Вт. Шумоизоляция — до 32 дБ. Светопропускание — около 80%.

Двухслойные окна рекомендованы для комнат, выходящих во двор, и для регионов с умеренным климатом. Для шумных магистралей и холодных регионов лучше выбирать трёхслойные конструкции.

## Технологии и комплектующие

Мы используем профильные системы Rehau, KBE, Veka и Montblanc. Стеклопакеты — энергосберегающие с напылением i-типа, с аргоном или криптоном внутри камер. Фурнитура — Roto, Maco, Siegenia. Возможна установка москитных сеток, детских замков и микропроветривания.

## Цены

Стоимость двухслойного окна «под ключ» (изготовление, доставка, демонтаж старого, монтаж нового) — от 12 000 ₽ за м². Цена зависит от размеров, профиля, типа стеклопакета и фурнитуры. Точная смета составляется после замера.
''')

# ========== okna_pvh_3sloy.md ==========
with open(f"{KB}/okna_pvh_3sloy.md", "w", encoding="utf-8") as f:
    f.write('''# Трёхслойные ПВХ-окна

Трёхслойные ПВХ-окна — это конструкция с трёхкамерным стеклопакетом (четыре стекла, три воздушные камеры). Такие окна дают максимальную тепло- и шумоизоляцию и рекомендованы для холодных регионов, шумных магистралей и панорамного остекления.

## Характеристики

Ширина профиля — 70–90 мм. Количество камер профиля — 5–7. Сопротивление теплопередаче — 0,58–0,72 м²·°C/Вт. Шумоизоляция — до 42 дБ. Светопропускание — около 72–76%.

Трёхслойные окна снижают теплопотери на 25–30% по сравнению с двухслойными. Это уменьшает расходы на отопление и защищает от уличного шума.

## Технологии

Используются профильные системы Rehau Geneo, Veka Softline, KBE Expert. Стеклопакеты — двухкамерные энергосберегающие с аргоновым заполнением и тёплой дистанционной рамкой. Возможно нанесение мультифункционального покрытия, защищающего от солнечного тепла летом.

## Цены

Стоимость трёхслойного окна «под ключ» — от 18 000 ₽ за м². Итоговая цена зависит от площади, профиля, стеклопакета и фурнитуры. При заказе от 5 окон — скидка 7%. При заказе от 10 окон — скидка 12%.
''')

# ========== balkonnye_bloki.md ==========
with open(f"{KB}/balkonnye_bloki.md", "w", encoding="utf-8") as f:
    f.write('''# Балконные блоки и остекление

RemStroy устанавливает балконные блоки (окно + балконная дверь) и выполняет остекление балконов и лоджий «под ключ».

## Балконные блоки

Балконный блок состоит из оконной части и балконной двери, объединённых в единую конструкцию. Мы изготавливаем блоки на двухслойных и трёхслойных ПВХ-профилях. Ширина двери — от 60 до 90 см, высота — до 230 см. Возможны варианты с антресольной фрамугой и поворотно-откидной створкой.

## Остекление балконов

Холодное остекление — алюминиевый профиль без утепления. Подходит для защиты от осадков и пыли. Цена — от 8 000 ₽ за м².

Тёплое остекление — ПВХ-профиль с двухкамерным или трёхкамерным стеклопакетом, с утеплением парапета и пола. Позволяет присоединить балкон к комнате. Цена — от 15 000 ₽ за м².

## Утепление и отделка

При тёплом остеклении мы утепляем парапет, пол и потолок минеральной ватой или пенополистиролом, монтируем пароизоляцию и выполняем чистовую отделку: пластиковые панели, ламинат, декоративная штукатурка. Срок работ — от 5 до 10 дней.
''')

# ========== tehnologii_montazha.md ==========
with open(f"{KB}/tehnologii_montazha.md", "w", encoding="utf-8") as f:
    f.write('''# Технологии монтажа окон и балконных блоков

RemStroy выполняет монтаж по ГОСТ 30971-2012 и по технологии «тёплого монтажа». Тип монтажа влияет на тепло- и шумоизоляцию и на срок службы конструкции.

## Стандартный монтаж по ГОСТ

Демонтаж старого блока, очистка проёма, установка нового блока на анкерные пластины или дюбели. Швы заполняются монтажной пеной, снаружи — паропроницаемая лента (ПСУЛ), внутри — пароизоляционная лента. Такой монтаж подходит для большинства жилых помещений.

## Тёплый монтаж

Технология с использованием пароизоляционных и паропроницаемых лент по всему периметру проёма. Швы заполняются двухкомпонентной пеной. Внутренний и наружный контуры герметизируются отдельно. Тёплый монтаж даёт максимальную защиту от продувания и промерзания, рекомендован для трёхслойных окон и панорамного остекления.

## Сроки и гарантия на монтаж

Стандартный монтаж одного окна — 2–3 часа. Балконный блок — 4–6 часов. Тёплый монтаж — на 30–50% дольше. Гарантия на монтажные работы — 3 года, на конструкцию — 5 лет.
''')

# ========== garantii_i_sroki.md ==========
with open(f"{KB}/garantii_i_sroki.md", "w", encoding="utf-8") as f:
    f.write('''# Гарантии и сроки RemStroy

## Гарантия на окна и балконные блоки

На профиль и стеклопакет — 5 лет. На фурнитуру — 3 года. На монтажные работы — 3 года. Гарантия действует при соблюдении правил эксплуатации: регулярная смазка фурнитуры, своевременная регулировка створок, отсутствие механических повреждений.

## Гарантия на ремонт квартир

На общестроительные работы — 2 года. На инженерные системы (электрика, сантехника) — 3 года. На чистовую отделку — 1 год. Гарантийные обязательства фиксируются в договоре и акте приёмки.

## Сроки выполнения

Замер — в течение 1–2 рабочих дней после заявки. Изготовление окон — 5–10 рабочих дней. Монтаж — 1 день. Ремонт квартиры — от 2 недель (косметический) до 8 месяцев (дизайнерский).

## Рекламации

При обнаружении дефекта в течение гарантийного срока оставьте заявку по телефону или через сайт. Выезд мастера — в течение 3 рабочих дней. Устранение дефекта — в течение 14 дней с момента подтверждения.
''')

# ========== ceny_i_usloviya.md ==========
with open(f"{KB}/ceny_i_usloviya.md", "w", encoding="utf-8") as f:
    f.write('''# Цены и условия работы RemStroy

## Окна и балконные блоки

Двухслойные ПВХ-окна — от 12 000 ₽ за м² «под ключ». Трёхслойные ПВХ-окна — от 18 000 ₽ за м² «под ключ». Балконный блок (окно + дверь) — от 25 000 ₽. Холодное остекление балкона — от 8 000 ₽ за м². Тёплое остекление балкона — от 15 000 ₽ за м².

## Ремонт квартир

Косметический ремонт — от 4 500 ₽ за м². Капитальный ремонт — от 9 000 ₽ за м². Дизайнерский ремонт — от 15 000 ₽ за м². Цены указаны за работу без стоимости чистовых материалов.

## Скидки и рассрочка

При заказе от 5 окон — скидка 7%. От 10 окон — скидка 12%. Пенсионерам — скидка 5% на монтаж. При комплексном заказе (ремонт + окна) — скидка 10% на окна.

Рассрочка на 6 или 12 месяцев без процентов при заказе от 100 000 ₽. Оформление — через банк-партнёр, требуется паспорт РФ.

## Оплата

Предоплата 30% после подписания договора. 40% — после доставки материалов. 30% — после приёмки работ. Безналичный расчёт для юридических лиц, наличные и карта — для физических.
''')

import os
print("✓ База знаний создана. Файлы:")
for name in sorted(os.listdir(KB)):
    size = os.path.getsize(f"{KB}/{name}")
    print(f"  {name} ({size} байт)")

✓ База знаний создана. Файлы:
  balkonnye_bloki.md (1858 байт)
  ceny_i_usloviya.md (1714 байт)
  garantii_i_sroki.md (1704 байт)
  okna_pvh_2sloy.md (2043 байт)
  okna_pvh_3sloy.md (1916 байт)
  remont_services.md (2558 байт)
  tehnologii_montazha.md (1855 байт)


In [ ]:
ST = "/content/static"

# ========== index.html ==========
with open(f"{ST}/index.html", "w", encoding="utf-8") as f:
    f.write('''<!DOCTYPE html>
<html lang="ru">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>RemStroy Assistant</title>
  <link rel="stylesheet" href="/style.css">
</head>
<body>

  <header class="header">
    <div class="logo">
      <svg width="32" height="32" viewBox="0 0 32 32" aria-hidden="true">
        <rect x="2" y="2" width="28" height="28" rx="6" fill="#2f3542"/>
        <path d="M8 22 L8 12 L13 8 L18 12 L18 22"
              stroke="#ffffff" stroke-width="2"
              fill="none" stroke-linejoin="round"/>
        <rect x="14" y="16" width="3" height="6" fill="#ffffff"/>
      </svg>
      <div class="logo-text">
        <span class="logo-title">RemStroy</span>
        <span class="logo-sub">Ассистент по ремонту и окнам</span>
      </div>
    </div>
    <div class="header-actions">
      <button id="btn-reindex" class="btn-ghost">Переиндексировать</button>
      <button id="btn-reset" class="btn-ghost">Новый диалог</button>
      <span id="status" class="status"></span>
    </div>
  </header>

  <main class="container">
    <section class="chat" id="chat">
      <div class="message assistant">
        <div class="bubble">
          Здравствуйте! Я ассистент RemStroy. Задайте вопрос о ремонте
          квартир, установке оконных или балконных блоков.
        </div>
      </div>
    </section>

    <section class="composer">
      <textarea id="question" rows="2"
        placeholder="Например: Сколько стоит установка трёхслойного окна?"></textarea>
      <button id="btn-send" class="btn-primary">Спросить</button>
    </section>
  </main>

  <script src="/app.js"></script>
</body>
</html>
''')

# ========== style.css ==========
with open(f"{ST}/style.css", "w", encoding="utf-8") as f:
    f.write('''* { box-sizing: border-box; }

html, body {
  margin: 0;
  padding: 0;
  background: #f5f6f8;
  color: #2f3542;
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
               "Helvetica Neue", Arial, sans-serif;
  font-size: 15px;
  line-height: 1.5;
}

.header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  padding: 14px 24px;
  background: #ffffff;
  border-bottom: 1px solid #e3e5e8;
}

.logo { display: flex; align-items: center; gap: 12px; }
.logo-text { display: flex; flex-direction: column; }
.logo-title { font-weight: 600; font-size: 16px; color: #2f3542; }
.logo-sub { font-size: 12px; color: #8a8f98; }
.header-actions { display: flex; align-items: center; gap: 10px; }
.status { font-size: 12px; color: #8a8f98; margin-left: 8px; }

.btn-primary, .btn-ghost {
  border-radius: 8px;
  padding: 8px 14px;
  font-size: 14px;
  cursor: pointer;
  border: 1px solid transparent;
  transition: background .15s ease, border-color .15s ease;
}

.btn-primary { background: #2f3542; color: #ffffff; }
.btn-primary:hover { background: #1f242e; }
.btn-primary:disabled { background: #a4a9b3; cursor: not-allowed; }

.btn-ghost {
  background: #ffffff;
  color: #2f3542;
  border-color: #d5d8dd;
}
.btn-ghost:hover { background: #f0f1f3; }

.container {
  max-width: 880px;
  margin: 0 auto;
  padding: 20px 24px 40px;
}

.chat {
  display: flex;
  flex-direction: column;
  gap: 16px;
  margin-bottom: 16px;
}

.message { display: flex; }
.message.user { justify-content: flex-end; }
.message.assistant { justify-content: flex-start; }

.bubble {
  max-width: 100%;
  padding: 12px 16px;
  border-radius: 12px;
  background: #ffffff;
  border: 1px solid #e3e5e8;
  white-space: pre-wrap;
  word-wrap: break-word;
}

.message.user .bubble {
  background: #eef0f3;
  border-color: #e0e3e7;
}

.chunks {
  margin-top: 12px;
  border-top: 1px dashed #dcdfe4;
  padding-top: 10px;
}

.chunks-title {
  font-size: 13px;
  font-weight: 600;
  color: #6b7280;
  margin-bottom: 8px;
  cursor: pointer;
  user-select: none;
}

.chunk {
  background: #fafbfc;
  border: 1px solid #e8eaee;
  border-radius: 8px;
  padding: 10px 12px;
  margin-bottom: 8px;
  font-size: 13px;
}

.chunk-head {
  display: flex;
  justify-content: space-between;
  gap: 8px;
  font-size: 12px;
  color: #8a8f98;
  margin-bottom: 6px;
}

.chunk-score { font-variant-numeric: tabular-nums; }
.chunk-text { color: #3b414d; white-space: pre-wrap; }

.composer {
  display: flex;
  gap: 10px;
  align-items: flex-end;
  position: sticky;
  bottom: 16px;
  background: #f5f6f8;
  padding-top: 8px;
}

#question {
  flex: 1;
  resize: vertical;
  min-height: 48px;
  max-height: 160px;
  padding: 12px 14px;
  border-radius: 10px;
  border: 1px solid #d5d8dd;
  background: #ffffff;
  font: inherit;
  color: #2f3542;
  outline: none;
}
#question:focus { border-color: #a4a9b3; }

.error {
  color: #b00020;
  background: #fdecef;
  border: 1px solid #f5c6cf;
  padding: 10px 12px;
  border-radius: 8px;
}

.loading { font-size: 13px; color: #8a8f98; }
''')

# ========== app.js ==========
with open(f"{ST}/app.js", "w", encoding="utf-8") as f:
    f.write('''"use strict";

const chatEl = document.getElementById("chat");
const questionEl = document.getElementById("question");
const sendBtn = document.getElementById("btn-send");
const reindexBtn = document.getElementById("btn-reindex");
const resetBtn = document.getElementById("btn-reset");
const statusEl = document.getElementById("status");

function escapeHtml(s) {
  return String(s)
    .replace(/&/g, "&amp;")
    .replace(/</g, "&lt;")
    .replace(/>/g, "&gt;");
}

function scrollToBottom() {
  window.scrollTo({ top: document.body.scrollHeight, behavior: "smooth" });
}

function appendUserMessage(text) {
  const wrap = document.createElement("div");
  wrap.className = "message user";
  wrap.innerHTML = `<div class="bubble">${escapeHtml(text)}</div>`;
  chatEl.appendChild(wrap);
  scrollToBottom();
}

function appendAssistantMessage(answer, chunks) {
  const wrap = document.createElement("div");
  wrap.className = "message assistant";

  const bubble = document.createElement("div");
  bubble.className = "bubble";

  const answerDiv = document.createElement("div");
  answerDiv.textContent = answer;
  bubble.appendChild(answerDiv);

  if (Array.isArray(chunks) && chunks.length > 0) {
    const details = document.createElement("div");
    details.className = "chunks";

    const title = document.createElement("div");
    title.className = "chunks-title";
    title.textContent = `Найденные фрагменты (${chunks.length}) — показать`;
    details.appendChild(title);

    const list = document.createElement("div");
    list.style.display = "none";

    chunks.forEach((c, i) => {
      const item = document.createElement("div");
      item.className = "chunk";

      const head = document.createElement("div");
      head.className = "chunk-head";
      head.innerHTML =
        `<span>#${i + 1} · ${escapeHtml(c.source || "unknown")}` +
        ` · чанк ${c.chunk ?? "-"}</span>` +
        `<span class="chunk-score">релевантность: ${c.score}</span>`;
      item.appendChild(head);

      const txt = document.createElement("div");
      txt.className = "chunk-text";
      txt.textContent = c.text;
      item.appendChild(txt);

      list.appendChild(item);
    });

    title.addEventListener("click", () => {
      const open = list.style.display !== "none";
      list.style.display = open ? "none" : "block";
      title.textContent =
        `Найденные фрагменты (${chunks.length}) — ` +
        (open ? "показать" : "скрыть");
    });

    details.appendChild(list);
    bubble.appendChild(details);
  }

  wrap.appendChild(bubble);
  chatEl.appendChild(wrap);
  scrollToBottom();
}

function appendError(text) {
  const wrap = document.createElement("div");
  wrap.className = "message assistant";
  wrap.innerHTML = `<div class="bubble error">${escapeHtml(text)}</div>`;
  chatEl.appendChild(wrap);
  scrollToBottom();
}

function appendLoading() {
  const wrap = document.createElement("div");
  wrap.className = "message assistant";
  wrap.innerHTML = `<div class="bubble loading">Ищу ответ…</div>`;
  chatEl.appendChild(wrap);
  scrollToBottom();
  return wrap;
}

function setBusy(isBusy) {
  sendBtn.disabled = isBusy;
  sendBtn.textContent = isBusy ? "Отправка…" : "Спросить";
}

async function refreshStatus() {
  try {
    const r = await fetch("/api/status");
    const data = await r.json();
    statusEl.textContent = `чанков в индексе: ${data.indexed_chunks}`;
  } catch (e) {
    statusEl.textContent = "";
  }
}

async function ask() {
  const question = questionEl.value.trim();
  if (!question) return;

  appendUserMessage(question);
  questionEl.value = "";
  setBusy(true);

  const loading = appendLoading();

  try {
    const resp = await fetch("/api/query", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ question }),
    });
    const data = await resp.json();
    loading.remove();

    if (data.error) {
      appendError("Ошибка: " + data.error);
    } else {
      appendAssistantMessage(data.answer || "", data.chunks || []);
    }
  } catch (e) {
    loading.remove();
    appendError("Не удалось получить ответ: " + e.message);
  } finally {
    setBusy(false);
    questionEl.focus();
  }
}

async function reindex() {
  reindexBtn.disabled = true;
  reindexBtn.textContent = "Индексация…";
  try {
    const r = await fetch("/api/reindex", { method: "POST" });
    const data = await r.json();
    if (data.error) {
      appendError("Ошибка индексации: " + data.error);
    } else {
      appendAssistantMessage(
        `База знаний проиндексирована. Чанков: ${data.indexed_chunks}.`, []
      );
      refreshStatus();
    }
  } catch (e) {
    appendError("Не удалось переиндексировать: " + e.message);
  } finally {
    reindexBtn.disabled = false;
    reindexBtn.textContent = "Переиндексировать";
  }
}

async function resetChat() {
  try { await fetch("/api/reset", { method: "POST" }); } catch (e) {}
  chatEl.innerHTML = "";
  appendAssistantMessage("Диалог сброшен. Задайте новый вопрос.", []);
}

sendBtn.addEventListener("click", ask);
questionEl.addEventListener("keydown", (e) => {
  if (e.key === "Enter" && !e.shiftKey) {
    e.preventDefault();
    ask();
  }
});
reindexBtn.addEventListener("click", reindex);
resetBtn.addEventListener("click", resetChat);

refreshStatus();
''')

import os
print("✓ Фронтенд создан. Файлы static/:")
for name in sorted(os.listdir(ST)):
    size = os.path.getsize(f"{ST}/{name}")
    print(f"  {name} ({size} байт)")

✓ Фронтенд создан. Файлы static/:
  app.js (5601 байт)
  index.html (1871 байт)
  style.css (3069 байт)


In [ ]:
import os
from google.colab import userdata

# 1. Сначала — ключ в окружение
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Ключ получен, длина:", len(os.environ["GEMINI_API_KEY"]))

# 2. Переход в рабочую директорию
os.chdir("/content")

# 3. На случай если модули уже загружены — сбрасываем кэш
import sys
for mod in ["config", "openai_client", "chroma_store", "rag_pipeline", "main"]:
    if mod in sys.modules:
        del sys.modules[mod]

# 4. Теперь импортируем — config прочитает ключ
import config
import openai_client
import chroma_store
import rag_pipeline

print("✓ Все модули импортируются без ошибок")
print("Модель генерации:", config.CHAT_MODEL)
print("Модель эмбеддингов:", config.EMBEDDING_MODEL)

Ключ получен, длина: 53
✓ Все модули импортируются без ошибок
Модель генерации: gemini-3.6-flash
Модель эмбеддингов: gemini-embedding-001


In [ ]:
import rag_pipeline

n = rag_pipeline.build_index()
print(f"\nГотово. Чанков в индексе: {n}")

[RAG] Проиндексировано чанков: 30

Готово. Чанков в индексе: 30


In [ ]:
import importlib
import rag_pipeline
importlib.reload(rag_pipeline)

result = rag_pipeline.rag_query("Сколько стоит трёхслойное окно?")

print("=== 5 НАЙДЕННЫХ ЧАНКОВ ===\n")
for i, c in enumerate(result["chunks"], 1):
    print(f"--- Чанк {i} | {c['source']} | релевантность: {c['score']} ---")
    preview = c["text"][:220].replace("\n", " ")
    print(preview, "...\n")

print("\n=== AI-ОТВЕТ ===\n")
print(result["answer"])

=== 5 НАЙДЕННЫХ ЧАНКОВ ===

--- Чанк 1 | okna_pvh_3sloy.md | релевантность: 0.8261 ---
Используются профильные системы Rehau Geneo, Veka Softline, KBE Expert. Стеклопакеты — двухкамерные энергосберегающие с аргоновым заполнением и тёплой дистанционной рамкой. Возможно нанесение мультифункционального покрыт ...

--- Чанк 2 | ceny_i_usloviya.md | релевантность: 0.8059 ---
Двухслойные ПВХ-окна — от 12 000 ₽ за м² «под ключ». Трёхслойные ПВХ-окна — от 18 000 ₽ за м² «под ключ». Балконный блок (окно + дверь) — от 25 000 ₽. Холодное остекление балкона — от 8 000 ₽ за м². Тёплое остекление бал ...

--- Чанк 3 | okna_pvh_3sloy.md | релевантность: 0.8024 ---
# Трёхслойные ПВХ-окна  Трёхслойные ПВХ-окна — это конструкция с трёхкамерным стеклопакетом (четыре стекла, три воздушные камеры). Такие окна дают максимальную тепло- и шумоизоляцию и рекомендованы для холодных регионов, ...

--- Чанк 4 | ceny_i_usloviya.md | релевантность: 0.7863 ---
# Цены и условия работы RemStroy  ## Окна и балконные бл

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print("✓ cloudflared установлен")

✓ cloudflared установлен


In [ ]:
import subprocess
import time
import os

os.chdir("/content")

# Запускаем наш сервер в фоне
server = subprocess.Popen(
    ["python", "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(3)
print("Сервер запущен (PID:", server.pid, ")")

# Запускаем туннель cloudflared в фоне
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(8)

print("Туннель запущен (PID:", tunnel.pid, ")")
print("\nИщите ссылку ниже (строка вида https://xxx-xxx.trycloudflare.com):\n")

# Читаем первые строки вывода туннеля, где появится публичный URL
for _ in range(40):
    line = tunnel.stdout.readline()
    if line:
        print(line.rstrip())
    if "trycloudflare.com" in line:
        break

Сервер запущен (PID: 66855 )
Туннель запущен (PID: 66871 )

Ищите ссылку ниже (строка вида https://xxx-xxx.trycloudflare.com):

2026-10-04T18:32:20Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-04T18:32:20Z INF Requesting new quick Tunnel on trycloudflare.com...


In [ ]:
for i in range(60):  # до 30 секунд
    line = tunnel.stdout.readline()
    if line:
        print(line.rstrip())
        if "trycloudflare.com" in line and "https://" in line:
            # Извлекаем URL
            for word in line.split():
                if word.startswith("https://") and "trycloudflare.com" in word:
                    found_url = word
                    break
            if found_url:
                break
    else:
        time.sleep(0.5)

if found_url:
    print("\n" + "=" * 60)
    print("ВАША ССЫЛКА:", found_url)
    print("=" * 60)
    print("\nОткройте её в новой вкладке браузера.")
else:
    print("\nСсылка пока не появилась. Подождите 10 сек и запустите ячейку снова.")

2026-10-04T18:32:22Z INF +--------------------------------------------------------------------------------------------+
2026-10-04T18:32:22Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-04T18:32:22Z INF |  https://indicate-interest-veterinary-manual.trycloudflare.com                             |

ВАША ССЫЛКА: https://indicate-interest-veterinary-manual.trycloudflare.com

Откройте её в новой вкладке браузера.


In [ ]:
print("Сервер работает:", server.poll() is None)
print("Туннель работает:", tunnel.poll() is None)

Сервер работает: True
Туннель работает: True


In [1]:
import os
from google.colab import userdata

# Проверяем ключ из Secrets
try:
    key = userdata.get("GEMINI_API_KEY")
    os.environ["GEMINI_API_KEY"] = key
    print("✓ GEMINI_API_KEY получен, длина:", len(key))
except Exception as e:
    print("❌ Ошибка получения ключа:", e)
    print("Проверьте, что в панели Secrets есть GEMINI_API_KEY с включённым Notebook access")

# Проверяем рабочую директорию
os.chdir("/content")
print("Рабочая директория:", os.getcwd())
print("Содержимое /content/:", sorted(os.listdir("/content")))

✓ GEMINI_API_KEY получен, длина: 53
Рабочая директория: /content
Содержимое /content/: ['.config', 'sample_data']


In [2]:
# ===== rag_pipeline.py =====
with open("/content/rag_pipeline.py", "w", encoding="utf-8") as f:
    f.write('''import os
import glob

from config import (
    KNOWLEDGE_DIR, CHUNK_MAX_PARAGRAPHS, CHUNK_OVERLAP, TOP_K,
    HISTORY_MAX_MESSAGES,
)
from openai_client import get_embedding, get_embeddings_batch, chat
from chroma_store import add_chunks, query_chunks, reset_collection, count

conversation_history = []


def smart_chunk(text, max_paragraphs=CHUNK_MAX_PARAGRAPHS, overlap=CHUNK_OVERLAP):
    paragraphs = [p.strip() for p in text.split("\\n\\n") if p.strip()]
    chunks = []
    i = 0
    while i < len(paragraphs):
        end = min(i + max_paragraphs, len(paragraphs))
        chunk = "\\n\\n".join(paragraphs[i:end])
        chunks.append(chunk)
        if end >= len(paragraphs):
            break
        i = end - overlap
    return chunks


def load_knowledge_files():
    files = []
    pattern = os.path.join(KNOWLEDGE_DIR, "*.md")
    for path in sorted(glob.glob(pattern)):
        with open(path, "r", encoding="utf-8") as fh:
            files.append((os.path.basename(path), fh.read()))
    return files


def build_index(verbose=True):
    reset_collection()
    files = load_knowledge_files()
    if not files:
        if verbose:
            print("[RAG] Папка knowledge_base пуста.")
        return 0

    ids, documents, metadatas, embeddings = [], [], [], []
    for filename, text in files:
        chunks = smart_chunk(text)
        for idx, chunk in enumerate(chunks):
            ids.append(f"{filename}::{idx}")
            documents.append(chunk)
            metadatas.append({"source": filename, "chunk": idx})

    BATCH = 64
    for start in range(0, len(documents), BATCH):
        batch = documents[start:start + BATCH]
        embeddings.extend(get_embeddings_batch(batch))

    add_chunks(ids, documents, embeddings, metadatas)
    if verbose:
        print(f"[RAG] Проиндексировано чанков: {len(ids)}")
    return len(ids)


def ensure_index():
    if count() == 0:
        build_index()


def search(query, n_results=TOP_K):
    query_emb = get_embedding(query)
    results = query_chunks(query_emb, n_results=n_results)
    docs = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    hits = []
    for doc, dist, meta in zip(docs, dists, metas):
        score = 1.0 - float(dist)
        hits.append({
            "text": doc,
            "source": meta.get("source", "unknown"),
            "chunk": meta.get("chunk", -1),
            "score": round(score, 4),
        })
    return hits


SYSTEM_PROMPT = (
    "Ты — ассистент компании RemStroy. Компания занимается ремонтом "
    "жилых квартир и установкой оконных и балконных блоков "
    "(2- и 3-слойных) по различным технологиям.\\n\\n"
    "ПРАВИЛА:\\n"
    "1. Отвечай ТОЛЬКО на последний заданный вопрос. "
    "Игнорируй предыдущие вопросы, если они не относятся к делу.\\n"
    "2. Не начинай ответ с фраз про отсутствие информации, "
    "если ответ на заданный вопрос есть в контексте.\\n"
    "3. Не упоминай темы, которых нет в текущем вопросе.\\n"
    "4. Если информации действительно нет — скажи об этом одной "
    "краткой фразой и остановись.\\n"
    "5. Отвечай на русском языке, по существу, без вводных слов."
)


def generate_answer(question, hits):
    context_parts = []
    for i, h in enumerate(hits, 1):
        context_parts.append(
            f"[Фрагмент {i} | источник: {h['source']}]\\n{h['text']}"
        )
    context = "\\n\\n---\\n\\n".join(context_parts)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Контекст:\\n{context}\\n\\nВопрос: {question}",
        },
    ]
    return chat(messages)


def rag_query(question):
    ensure_index()
    hits = search(question, n_results=TOP_K)
    answer = generate_answer(question, hits)
    conversation_history.append({"role": "user", "content": question})
    conversation_history.append({"role": "assistant", "content": answer})
    if len(conversation_history) > HISTORY_MAX_MESSAGES:
        conversation_history[:] = conversation_history[-HISTORY_MAX_MESSAGES:]
    return {"chunks": hits, "answer": answer}


def reset_history():
    conversation_history.clear()
''')

# ===== main.py =====
with open("/content/main.py", "w", encoding="utf-8") as f:
    f.write('''import json
import os
import http.server
import socketserver

from config import HOST, PORT
from rag_pipeline import rag_query, build_index, reset_history
from chroma_store import count

STATIC_DIR = "/content/static"


class Handler(http.server.BaseHTTPRequestHandler):
    def _send_json(self, payload, status=200):
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _read_json(self):
        length = int(self.headers.get("Content-Length", 0))
        if length == 0:
            return {}
        raw = self.rfile.read(length)
        try:
            return json.loads(raw.decode("utf-8"))
        except Exception:
            return {}

    def _serve_static(self, rel_path):
        safe_rel = rel_path.lstrip("/").replace("..", "")
        full = os.path.join(STATIC_DIR, safe_rel)
        if not os.path.isfile(full):
            self.send_error(404, "Not found")
            return
        ext = os.path.splitext(full)[1].lower()
        content_type = {
            ".html": "text/html; charset=utf-8",
            ".css":  "text/css; charset=utf-8",
            ".js":   "application/javascript; charset=utf-8",
            ".svg":  "image/svg+xml",
        }.get(ext, "application/octet-stream")
        with open(full, "rb") as fh:
            data = fh.read()
        self.send_response(200)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if self.path in ("/", "/index.html"):
            self._serve_static("index.html")
        elif self.path == "/api/status":
            self._send_json({"indexed_chunks": count()})
        else:
            self._serve_static(self.path)

    def do_POST(self):
        if self.path == "/api/query":
            data = self._read_json()
            question = (data.get("question") or "").strip()
            if not question:
                self._send_json({"error": "Пустой вопрос"}, status=400)
                return
            try:
                result = rag_query(question)
                self._send_json(result)
            except Exception as e:
                self._send_json({"error": str(e)}, status=500)
        elif self.path == "/api/reindex":
            try:
                n = build_index()
                reset_history()
                self._send_json({"indexed_chunks": n})
            except Exception as e:
                self._send_json({"error": str(e)}, status=500)
        elif self.path == "/api/reset":
            reset_history()
            self._send_json({"ok": True})
        else:
            self.send_error(404, "Not found")

    def log_message(self, fmt, *args):
        return


def run():
    with socketserver.TCPServer((HOST, PORT), Handler) as httpd:
        print(f"[RemStroy] Сервер запущен на порту {PORT}")
        httpd.serve_forever()


if __name__ == "__main__":
    run()
''')

print("✓ rag_pipeline.py и main.py созданы")
print(sorted(os.listdir("/content")))

✓ rag_pipeline.py и main.py созданы
['.config', 'main.py', 'rag_pipeline.py', 'sample_data']


In [3]:
import os
print("Файлы в /content/:")
for f in sorted(os.listdir("/content")):
    if f.endswith(".py") or f in ("knowledge_base", "static"):
        print(" ", f)

Файлы в /content/:
  main.py
  rag_pipeline.py


In [4]:
# ===== config.py =====
with open("/content/config.py", "w", encoding="utf-8") as f:
    f.write('''import os

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
CHAT_MODEL = "gemini-3.6-flash"
EMBEDDING_MODEL = "gemini-embedding-001"

TEMPERATURE = 0.3
MAX_TOKENS = 500

CHUNK_MAX_PARAGRAPHS = 3
CHUNK_OVERLAP = 1
TOP_K = 5

HISTORY_MAX_MESSAGES = 10

CHROMA_PATH = "/content/chroma_db"
COLLECTION_NAME = "remstroy_knowledge"

KNOWLEDGE_DIR = "/content/knowledge_base"

HOST = "0.0.0.0"
PORT = 8000
''')

# ===== openai_client.py =====
with open("/content/openai_client.py", "w", encoding="utf-8") as f:
    f.write('''from openai import OpenAI
from config import (
    GEMINI_API_KEY, BASE_URL, CHAT_MODEL, EMBEDDING_MODEL,
    TEMPERATURE, MAX_TOKENS,
)

_client = OpenAI(api_key=GEMINI_API_KEY, base_url=BASE_URL)


def get_embedding(text):
    response = _client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


def get_embeddings_batch(texts):
    response = _client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


def chat(messages):
    response = _client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content
''')

# ===== chroma_store.py =====
with open("/content/chroma_store.py", "w", encoding="utf-8") as f:
    f.write('''import chromadb
from config import CHROMA_PATH, COLLECTION_NAME

_client = chromadb.PersistentClient(path=CHROMA_PATH)
_collection = _client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)


def get_collection():
    return _collection


def reset_collection():
    global _collection
    try:
        _client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    _collection = _client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )
    return _collection


def add_chunks(ids, documents, embeddings, metadatas):
    _collection.add(
        ids=ids, documents=documents,
        embeddings=embeddings, metadatas=metadatas,
    )


def query_chunks(query_embedding, n_results=5):
    return _collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "distances", "metadatas"],
    )


def count():
    return _collection.count()


def get_all_documents():
    """Возвращает все чанки — для построения BM25-индекса."""
    result = _collection.get(include=["documents", "metadatas"])
    return result["ids"], result["documents"], result["metadatas"]
''')

import os
print("✓ Файлы созданы:")
for f in sorted(os.listdir("/content")):
    if f.endswith(".py"):
        print(" ", f)

✓ Файлы созданы:
  chroma_store.py
  config.py
  main.py
  openai_client.py
  rag_pipeline.py


In [5]:
import os
KB = "/content/knowledge_base"
os.makedirs(KB, exist_ok=True)

# ===== remont_services.md =====
with open(f"{KB}/remont_services.md", "w", encoding="utf-8") as f:
    f.write('''# Ремонт квартир — услуги RemStroy

Компания RemStroy выполняет ремонт жилых квартир любой сложности. Мы работаем с новостройками и вторичным жильём, с полной или частичной отделкой. Все работы выполняются по договору с фиксированной сметой.

## Виды ремонта

Косметический ремонт — обновление отделки без вмешательства в инженерные системы и планировку. Включает покраску стен, поклейку обоев, замену напольного покрытия, мелкий ремонт потолков. Срок выполнения — от 2 до 4 недель в зависимости от площади квартиры.

Капитальный ремонт — полная замена отделки, инженерных коммуникаций (электрика, сантехника), выравнивание стен и полов, замена окон и дверей. Срок выполнения — от 2 до 4 месяцев.

Дизайнерский ремонт — ремонт по индивидуальному проекту дизайнера, с перепланировкой, авторским надзором и использованием премиальных материалов. Срок выполнения — от 4 до 8 месяцев.

## Этапы работ

Каждый ремонт проходит через фиксированные этапы: замер и проектирование, демонтаж, черновые работы, монтаж инженерных систем, чистовая отделка, финальная приёмка. На каждом этапе заказчик получает фотоотчёт и может внести правки до начала следующего этапа.

## Что входит в стоимость

В стоимость ремонта входят работы, черновые материалы и вывоз строительного мусора. Чистовые материалы (обои, плитка, краска, напольные покрытия) заказчик выбирает и оплачивает отдельно либо заказывает через нас с наценкой 10%.
''')

# ===== okna_pvh_2sloy.md =====
with open(f"{KB}/okna_pvh_2sloy.md", "w", encoding="utf-8") as f:
    f.write('''# Двухслойные ПВХ-окна

Двухслойные ПВХ-окна — это конструкция с двухкамерным стеклопакетом (три стекла, две воздушные камеры). Такие окна обеспечивают базовую тепло- и шумоизоляцию и подходят для большинства жилых помещений в средней полосе.

## Характеристики

Ширина профиля — 58–70 мм. Количество камер профиля — 3–5. Сопротивление теплопередаче стеклопакета — 0,47–0,52 м²·°C/Вт. Шумоизоляция — до 32 дБ. Светопропускание — около 80%.

Двухслойные окна рекомендованы для комнат, выходящих во двор, и для регионов с умеренным климатом. Для шумных магистралей и холодных регионов лучше выбирать трёхслойные конструкции.

## Технологии и комплектующие

Мы используем профильные системы Rehau, KBE, Veka и Montblanc. Стеклопакеты — энергосберегающие с напылением i-типа, с аргоном или криптоном внутри камер. Фурнитура — Roto, Maco, Siegenia. Возможна установка москитных сеток, детских замков и микропроветривания.

## Цены

Стоимость двухслойного окна «под ключ» (изготовление, доставка, демонтаж старого, монтаж нового) — от 12 000 ₽ за м². Цена зависит от размеров, профиля, типа стеклопакета и фурнитуры. Точная смета составляется после замера.
''')

# ===== okna_pvh_3sloy.md =====
with open(f"{KB}/okna_pvh_3sloy.md", "w", encoding="utf-8") as f:
    f.write('''# Трёхслойные ПВХ-окна

Трёхслойные ПВХ-окна — это конструкция с трёхкамерным стеклопакетом (четыре стекла, три воздушные камеры). Такие окна дают максимальную тепло- и шумоизоляцию и рекомендованы для холодных регионов, шумных магистралей и панорамного остекления.

## Характеристики

Ширина профиля — 70–90 мм. Количество камер профиля — 5–7. Сопротивление теплопередаче — 0,58–0,72 м²·°C/Вт. Шумоизоляция — до 42 дБ. Светопропускание — около 72–76%.

Трёхслойные окна снижают теплопотери на 25–30% по сравнению с двухслойными. Это уменьшает расходы на отопление и защищает от уличного шума.

## Технологии

Используются профильные системы Rehau Geneo, Veka Softline, KBE Expert. Стеклопакеты — двухкамерные энергосберегающие с аргоновым заполнением и тёплой дистанционной рамкой. Возможно нанесение мультифункционального покрытия, защищающего от солнечного тепла летом.

## Цены

Стоимость трёхслойного окна «под ключ» — от 18 000 ₽ за м². Итоговая цена зависит от площади, профиля, стеклопакета и фурнитуры. При заказе от 5 окон — скидка 7%. При заказе от 10 окон — скидка 12%.
''')

# ===== balkonnye_bloki.md =====
with open(f"{KB}/balkonnye_bloki.md", "w", encoding="utf-8") as f:
    f.write('''# Балконные блоки и остекление

RemStroy устанавливает балконные блоки (окно + балконная дверь) и выполняет остекление балконов и лоджий «под ключ».

## Балконные блоки

Балконный блок состоит из оконной части и балконной двери, объединённых в единую конструкцию. Мы изготавливаем блоки на двухслойных и трёхслойных ПВХ-профилях. Ширина двери — от 60 до 90 см, высота — до 230 см. Возможны варианты с антресольной фрамугой и поворотно-откидной створкой.

## Остекление балконов

Холодное остекление — алюминиевый профиль без утепления. Подходит для защиты от осадков и пыли. Цена — от 8 000 ₽ за м².

Тёплое остекление — ПВХ-профиль с двухкамерным или трёхкамерным стеклопакетом, с утеплением парапета и пола. Позволяет присоединить балкон к комнате. Цена — от 15 000 ₽ за м².

## Утепление и отделка

При тёплом остеклении мы утепляем парапет, пол и потолок минеральной ватой или пенополистиролом, монтируем пароизоляцию и выполняем чистовую отделку: пластиковые панели, ламинат, декоративная штукатурка. Срок работ — от 5 до 10 дней.
''')

# ===== tehnologii_montazha.md =====
with open(f"{KB}/tehnologii_montazha.md", "w", encoding="utf-8") as f:
    f.write('''# Технологии монтажа окон и балконных блоков

RemStroy выполняет монтаж по ГОСТ 30971-2012 и по технологии «тёплого монтажа». Тип монтажа влияет на тепло- и шумоизоляцию и на срок службы конструкции.

## Стандартный монтаж по ГОСТ

Демонтаж старого блока, очистка проёма, установка нового блока на анкерные пластины или дюбели. Швы заполняются монтажной пеной, снаружи — паропроницаемая лента (ПСУЛ), внутри — пароизоляционная лента. Такой монтаж подходит для большинства жилых помещений.

## Тёплый монтаж

Технология с использованием пароизоляционных и паропроницаемых лент по всему периметру проёма. Швы заполняются двухкомпонентной пеной. Внутренний и наружный контуры герметизируются отдельно. Тёплый монтаж даёт максимальную защиту от продувания и промерзания, рекомендован для трёхслойных окон и панорамного остекления.

## Сроки и гарантия на монтаж

Стандартный монтаж одного окна — 2–3 часа. Балконный блок — 4–6 часов. Тёплый монтаж — на 30–50% дольше. Гарантия на монтажные работы — 3 года, на конструкцию — 5 лет.
''')

# ===== garantii_i_sroki.md =====
with open(f"{KB}/garantii_i_sroki.md", "w", encoding="utf-8") as f:
    f.write('''# Гарантии и сроки RemStroy

## Гарантия на окна и балконные блоки

На профиль и стеклопакет — 5 лет. На фурнитуру — 3 года. На монтажные работы — 3 года. Гарантия действует при соблюдении правил эксплуатации: регулярная смазка фурнитуры, своевременная регулировка створок, отсутствие механических повреждений.

## Гарантия на ремонт квартир

На общестроительные работы — 2 года. На инженерные системы (электрика, сантехника) — 3 года. На чистовую отделку — 1 год. Гарантийные обязательства фиксируются в договоре и акте приёмки.

## Сроки выполнения

Замер — в течение 1–2 рабочих дней после заявки. Изготовление окон — 5–10 рабочих дней. Монтаж — 1 день. Ремонт квартиры — от 2 недель (косметический) до 8 месяцев (дизайнерский).

## Рекламации

При обнаружении дефекта в течение гарантийного срока оставьте заявку по телефону или через сайт. Выезд мастера — в течение 3 рабочих дней. Устранение дефекта — в течение 14 дней с момента подтверждения.
''')

# ===== ceny_i_usloviya.md =====
with open(f"{KB}/ceny_i_usloviya.md", "w", encoding="utf-8") as f:
    f.write('''# Цены и условия работы RemStroy

## Окна и балконные блоки

Двухслойные ПВХ-окна — от 12 000 ₽ за м² «под ключ». Трёхслойные ПВХ-окна — от 18 000 ₽ за м² «под ключ». Балконный блок (окно + дверь) — от 25 000 ₽. Холодное остекление балкона — от 8 000 ₽ за м². Тёплое остекление балкона — от 15 000 ₽ за м².

## Ремонт квартир

Косметический ремонт — от 4 500 ₽ за м². Капитальный ремонт — от 9 000 ₽ за м². Дизайнерский ремонт — от 15 000 ₽ за м². Цены указаны за работу без стоимости чистовых материалов.

## Скидки и рассрочка

При заказе от 5 окон — скидка 7%. От 10 окон — скидка 12%. Пенсионерам — скидка 5% на монтаж. При комплексном заказе (ремонт + окна) — скидка 10% на окна.

Рассрочка на 6 или 12 месяцев без процентов при заказе от 100 000 ₽. Оформление — через банк-партнёр, требуется паспорт РФ.

## Оплата

Предоплата 30% после подписания договора. 40% — после доставки материалов. 30% — после приёмки работ. Безналичный расчёт для юридических лиц, наличные и карта — для физических.
''')

print("✓ База знаний создана:")
for name in sorted(os.listdir(KB)):
    print(f"  {name} ({os.path.getsize(f'{KB}/{name}')} байт)")

✓ База знаний создана:
  balkonnye_bloki.md (1858 байт)
  ceny_i_usloviya.md (1714 байт)
  garantii_i_sroki.md (1704 байт)
  okna_pvh_2sloy.md (2043 байт)
  okna_pvh_3sloy.md (1916 байт)
  remont_services.md (2558 байт)
  tehnologii_montazha.md (1855 байт)


In [6]:
import os
ST = "/content/static"
os.makedirs(ST, exist_ok=True)

# ===== index.html =====
with open(f"{ST}/index.html", "w", encoding="utf-8") as f:
    f.write('''<!DOCTYPE html>
<html lang="ru">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>RemStroy RAG Hybrid</title>
  <link rel="stylesheet" href="/style.css">
</head>
<body>

  <header class="header">
    <div class="logo">
      <svg width="32" height="32" viewBox="0 0 32 32" aria-hidden="true">
        <rect x="2" y="2" width="28" height="28" rx="6" fill="#2f3542"/>
        <path d="M8 22 L8 12 L13 8 L18 12 L18 22"
              stroke="#ffffff" stroke-width="2"
              fill="none" stroke-linejoin="round"/>
        <rect x="14" y="16" width="3" height="6" fill="#ffffff"/>
      </svg>
      <div class="logo-text">
        <span class="logo-title">RemStroy</span>
        <span class="logo-sub">Ассистент · Hybrid RAG</span>
      </div>
    </div>
    <div class="header-actions">
      <button id="btn-reindex" class="btn-ghost">Переиндексировать</button>
      <button id="btn-reset" class="btn-ghost">Новый диалог</button>
      <span id="status" class="status"></span>
    </div>
  </header>

  <main class="container">
    <section class="chat" id="chat">
      <div class="message assistant">
        <div class="bubble">
          Здравствуйте! Я ассистент RemStroy. Задайте вопрос о ремонте
          квартир, установке оконных или балконных блоков.
        </div>
      </div>
    </section>

    <section class="composer">
      <textarea id="question" rows="2"
        placeholder="Например: Сколько стоит установка трёхслойного окна?"></textarea>
      <button id="btn-send" class="btn-primary">Спросить</button>
    </section>
  </main>

  <script src="/app.js"></script>
</body>
</html>
''')

# ===== style.css =====
with open(f"{ST}/style.css", "w", encoding="utf-8") as f:
    f.write('''* { box-sizing: border-box; }

html, body {
  margin: 0;
  padding: 0;
  background: #f5f6f8;
  color: #2f3542;
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
               "Helvetica Neue", Arial, sans-serif;
  font-size: 15px;
  line-height: 1.5;
}

.header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  padding: 14px 24px;
  background: #ffffff;
  border-bottom: 1px solid #e3e5e8;
}

.logo { display: flex; align-items: center; gap: 12px; }
.logo-text { display: flex; flex-direction: column; }
.logo-title { font-weight: 600; font-size: 16px; color: #2f3542; }
.logo-sub { font-size: 12px; color: #8a8f98; }
.header-actions { display: flex; align-items: center; gap: 10px; }
.status { font-size: 12px; color: #8a8f98; margin-left: 8px; }

.btn-primary, .btn-ghost {
  border-radius: 8px;
  padding: 8px 14px;
  font-size: 14px;
  cursor: pointer;
  border: 1px solid transparent;
  transition: background .15s ease, border-color .15s ease;
}

.btn-primary { background: #2f3542; color: #ffffff; }
.btn-primary:hover { background: #1f242e; }
.btn-primary:disabled { background: #a4a9b3; cursor: not-allowed; }

.btn-ghost {
  background: #ffffff;
  color: #2f3542;
  border-color: #d5d8dd;
}
.btn-ghost:hover { background: #f0f1f3; }

.container {
  max-width: 880px;
  margin: 0 auto;
  padding: 20px 24px 40px;
}

.chat {
  display: flex;
  flex-direction: column;
  gap: 16px;
  margin-bottom: 16px;
}

.message { display: flex; }
.message.user { justify-content: flex-end; }
.message.assistant { justify-content: flex-start; }

.bubble {
  max-width: 100%;
  padding: 12px 16px;
  border-radius: 12px;
  background: #ffffff;
  border: 1px solid #e3e5e8;
  white-space: pre-wrap;
  word-wrap: break-word;
}

.message.user .bubble {
  background: #eef0f3;
  border-color: #e0e3e7;
}

/* === Блок «Логика работы RAG» === */
.rag-logic {
  margin-top: 14px;
  border-top: 1px dashed #dcdfe4;
  padding-top: 12px;
}

.rag-logic-title {
  font-size: 13px;
  font-weight: 600;
  color: #6b7280;
  margin-bottom: 10px;
  cursor: pointer;
  user-select: none;
}

.rag-section {
  margin-bottom: 14px;
}

.rag-section-title {
  font-size: 12px;
  font-weight: 600;
  color: #4b5563;
  text-transform: uppercase;
  letter-spacing: 0.04em;
  margin-bottom: 6px;
  display: flex;
  align-items: center;
  gap: 8px;
}

.rag-badge {
  display: inline-block;
  font-size: 11px;
  font-weight: 500;
  padding: 2px 8px;
  border-radius: 10px;
  background: #eef0f3;
  color: #4b5563;
  text-transform: none;
  letter-spacing: 0;
}

.rag-badge.embed { background: #e3efff; color: #1e40af; }
.rag-badge.keyword { background: #fef3e3; color: #92400e; }
.rag-badge.final { background: #e6f4ea; color: #065f46; }

.keywords {
  display: flex;
  flex-wrap: wrap;
  gap: 6px;
  margin-top: 4px;
}

.keyword-tag {
  font-size: 12px;
  background: #fef3e3;
  color: #92400e;
  padding: 3px 10px;
  border-radius: 12px;
  font-family: "SF Mono", Consolas, monospace;
}

.chunk {
  background: #fafbfc;
  border: 1px solid #e8eaee;
  border-radius: 8px;
  padding: 10px 12px;
  margin-bottom: 8px;
  font-size: 13px;
}

.chunk-head {
  display: flex;
  justify-content: space-between;
  gap: 8px;
  font-size: 12px;
  color: #8a8f98;
  margin-bottom: 6px;
}

.chunk-score { font-variant-numeric: tabular-nums; }
.chunk-text { color: #3b414d; white-space: pre-wrap; }

.composer {
  display: flex;
  gap: 10px;
  align-items: flex-end;
  position: sticky;
  bottom: 16px;
  background: #f5f6f8;
  padding-top: 8px;
}

#question {
  flex: 1;
  resize: vertical;
  min-height: 48px;
  max-height: 160px;
  padding: 12px 14px;
  border-radius: 10px;
  border: 1px solid #d5d8dd;
  background: #ffffff;
  font: inherit;
  color: #2f3542;
  outline: none;
}
#question:focus { border-color: #a4a9b3; }

.error {
  color: #b00020;
  background: #fdecef;
  border: 1px solid #f5c6cf;
  padding: 10px 12px;
  border-radius: 8px;
}

.loading { font-size: 13px; color: #8a8f98; }
''')

print("✓ index.html и style.css созданы")

✓ index.html и style.css созданы


In [7]:
ST = "/content/static"

with open(f"{ST}/app.js", "w", encoding="utf-8") as f:
    f.write('''"use strict";

const chatEl = document.getElementById("chat");
const questionEl = document.getElementById("question");
const sendBtn = document.getElementById("btn-send");
const reindexBtn = document.getElementById("btn-reindex");
const resetBtn = document.getElementById("btn-reset");
const statusEl = document.getElementById("status");

function escapeHtml(s) {
  return String(s)
    .replace(/&/g, "&amp;")
    .replace(/</g, "&lt;")
    .replace(/>/g, "&gt;");
}

function scrollToBottom() {
  window.scrollTo({ top: document.body.scrollHeight, behavior: "smooth" });
}

function appendUserMessage(text) {
  const wrap = document.createElement("div");
  wrap.className = "message user";
  wrap.innerHTML = `<div class="bubble">${escapeHtml(text)}</div>`;
  chatEl.appendChild(wrap);
  scrollToBottom();
}

function renderChunks(titleText, badgeClass, badgeLabel, chunks) {
  if (!Array.isArray(chunks) || chunks.length === 0) return null;

  const section = document.createElement("div");
  section.className = "rag-section";

  const title = document.createElement("div");
  title.className = "rag-section-title";
  title.innerHTML = `${titleText} <span class="rag-badge ${badgeClass}">${badgeLabel}</span>`;
  section.appendChild(title);

  chunks.forEach((c, i) => {
    const item = document.createElement("div");
    item.className = "chunk";

    const head = document.createElement("div");
    head.className = "chunk-head";
    const scoreLabel = c.score !== undefined
      ? `score: ${c.score}`
      : (c.bm25_score !== undefined ? `BM25: ${c.bm25_score}` : "");
    head.innerHTML =
      `<span>#${i + 1} · ${escapeHtml(c.source || "unknown")}` +
      ` · чанк ${c.chunk ?? "-"}</span>` +
      `<span class="chunk-score">${scoreLabel}</span>`;
    item.appendChild(head);

    const txt = document.createElement("div");
    txt.className = "chunk-text";
    txt.textContent = c.text;
    item.appendChild(txt);

    section.appendChild(item);
  });

  return section;
}

function appendAssistantMessage(answer, data) {
  const wrap = document.createElement("div");
  wrap.className = "message assistant";

  const bubble = document.createElement("div");
  bubble.className = "bubble";

  const answerDiv = document.createElement("div");
  answerDiv.textContent = answer;
  bubble.appendChild(answerDiv);

  // === Блок «Логика работы RAG» ===
  const ragBox = document.createElement("div");
  ragBox.className = "rag-logic";

  const ragTitle = document.createElement("div");
  ragTitle.className = "rag-logic-title";
  ragTitle.textContent = "▼ Логика работы RAG";
  ragBox.appendChild(ragTitle);

  const ragContent = document.createElement("div");

  // 1. Ключевые слова
  if (Array.isArray(data.keywords) && data.keywords.length > 0) {
    const kwSection = document.createElement("div");
    kwSection.className = "rag-section";
    kwSection.innerHTML = `
      <div class="rag-section-title">
        Ключевые слова из вопроса
        <span class="rag-badge keyword">без LLM</span>
      </div>
      <div class="keywords">
        ${data.keywords.map(k => `<span class="keyword-tag">${escapeHtml(k)}</span>`).join("")}
      </div>`;
    ragContent.appendChild(kwSection);
  }

  // 2. Embedding-поиск
  const embedSection = renderChunks(
    "Embedding-поиск (топ-5)",
    "embed", "cosine",
    data.embedding_chunks || data.chunks || []
  );
  if (embedSection) ragContent.appendChild(embedSection);

  // 3. Keyword-поиск (BM25)
  const kwChunksSection = renderChunks(
    "Поиск по ключевым словам (BM25)",
    "keyword", "BM25",
    data.keyword_chunks || []
  );
  if (kwChunksSection) ragContent.appendChild(kwChunksSection);

  // 4. Финальный набор (после RRF)
  if (Array.isArray(data.final_chunks) && data.final_chunks.length > 0) {
    const finalSection = renderChunks(
      "Финальный набор после объединения (RRF)",
      "final", "→ в LLM",
      data.final_chunks
    );
    if (finalSection) ragContent.appendChild(finalSection);
  }

  ragBox.appendChild(ragContent);
  bubble.appendChild(ragBox);

  wrap.appendChild(bubble);
  chatEl.appendChild(wrap);
  scrollToBottom();

  // Тоггл сворачивания
  ragTitle.addEventListener("click", () => {
    const open = ragContent.style.display !== "none";
    ragContent.style.display = open ? "none" : "block";
    ragTitle.textContent = (open ? "▶" : "▼") + " Логика работы RAG";
  });
}

function appendError(text) {
  const wrap = document.createElement("div");
  wrap.className = "message assistant";
  wrap.innerHTML = `<div class="bubble error">${escapeHtml(text)}</div>`;
  chatEl.appendChild(wrap);
  scrollToBottom();
}

function appendLoading() {
  const wrap = document.createElement("div");
  wrap.className = "message assistant";
  wrap.innerHTML = `<div class="bubble loading">Ищу ответ…</div>`;
  chatEl.appendChild(wrap);
  scrollToBottom();
  return wrap;
}

function setBusy(isBusy) {
  sendBtn.disabled = isBusy;
  sendBtn.textContent = isBusy ? "Отправка…" : "Спросить";
}

async function refreshStatus() {
  try {
    const r = await fetch("/api/status");
    const data = await r.json();
    statusEl.textContent = `чанков в индексе: ${data.indexed_chunks}`;
  } catch (e) {
    statusEl.textContent = "";
  }
}

async function ask() {
  const question = questionEl.value.trim();
  if (!question) return;

  appendUserMessage(question);
  questionEl.value = "";
  setBusy(true);

  const loading = appendLoading();

  try {
    const resp = await fetch("/api/query", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ question }),
    });
    const data = await resp.json();
    loading.remove();

    if (data.error) {
      appendError("Ошибка: " + data.error);
    } else {
      appendAssistantMessage(data.answer || "", data);
    }
  } catch (e) {
    loading.remove();
    appendError("Не удалось получить ответ: " + e.message);
  } finally {
    setBusy(false);
    questionEl.focus();
  }
}

async function reindex() {
  reindexBtn.disabled = true;
  reindexBtn.textContent = "Индексация…";
  try {
    const r = await fetch("/api/reindex", { method: "POST" });
    const data = await r.json();
    if (data.error) {
      appendError("Ошибка индексации: " + data.error);
    } else {
      appendAssistantMessage(
        `База знаний проиндексирована. Чанков: ${data.indexed_chunks}.`,
        { chunks: [] }
      );
      refreshStatus();
    }
  } catch (e) {
    appendError("Не удалось переиндексировать: " + e.message);
  } finally {
    reindexBtn.disabled = false;
    reindexBtn.textContent = "Переиндексировать";
  }
}

async function resetChat() {
  try { await fetch("/api/reset", { method: "POST" }); } catch (e) {}
  chatEl.innerHTML = "";
  appendAssistantMessage("Диалог сброшен. Задайте новый вопрос.", { chunks: [] });
}

sendBtn.addEventListener("click", ask);
questionEl.addEventListener("keydown", (e) => {
  if (e.key === "Enter" && !e.shiftKey) {
    e.preventDefault();
    ask();
  }
});
reindexBtn.addEventListener("click", reindex);
resetBtn.addEventListener("click", resetChat);

refreshStatus();
''')

import os
print("✓ app.js создан")
for name in sorted(os.listdir(ST)):
    print(f"  {name} ({os.path.getsize(f'{ST}/{name}')} байт)")

✓ app.js создан
  app.js (7513 байт)
  index.html (1852 байт)
  style.css (4021 байт)


In [8]:

import os
from google.colab import userdata

# Ключ уже должен быть в окружении, но подстрахуемся
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.chdir("/content")

# Сбрасываем кэш модулей на случай повторного запуска
import sys
for mod in ["config", "openai_client", "chroma_store", "rag_pipeline", "main"]:
    if mod in sys.modules:
        del sys.modules[mod]

# Импортируем
import config
import openai_client
import chroma_store
import rag_pipeline

print("✓ Импорты прошли")
print("Модель генерации:", config.CHAT_MODEL)
print("Модель эмбеддингов:", config.EMBEDDING_MODEL)
print("Чанков в индексе:", chroma_store.count())

# Индексируем
n = rag_pipeline.build_index()
print(f"\n✓ Проиндексировано: {n} чанков")

ModuleNotFoundError: No module named 'chromadb'

In [9]:
!pip install -q openai chromadb
print("✓ Установка завершена")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [1]:
import os
from google.colab import userdata

# Ключ в окружение
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.chdir("/content")

# Импорты
import config
import openai_client
import chroma_store
import rag_pipeline

print("✓ Импорты прошли")
print("Модель:", config.CHAT_MODEL)
print("Чанков в индексе:", chroma_store.count())

# Индексируем
n = rag_pipeline.build_index()
print(f"✓ Проиндексировано: {n} чанков")

# Тестовый запрос
result = rag_pipeline.rag_query("Сколько стоит трёхслойное окно?")
print("\n=== 5 ЧАНКОВ ===")
for i, c in enumerate(result["chunks"], 1):
    print(f"{i}. {c['source']} · score: {c['score']}")
print("\n=== ОТВЕТ ===\n", result["answer"])

✓ Импорты прошли
Модель: gemini-3.6-flash
Чанков в индексе: 0
[RAG] Проиндексировано чанков: 30
✓ Проиндексировано: 30 чанков

=== 5 ЧАНКОВ ===
1. okna_pvh_3sloy.md · score: 0.8261
2. ceny_i_usloviya.md · score: 0.8059
3. okna_pvh_3sloy.md · score: 0.8024
4. ceny_i_usloviya.md · score: 0.7863
5. okna_pvh_3sloy.md · score: 0.7837

=== ОТВЕТ ===
 Трёхслойное окно «под ключ» стоит от 18 000 ₽ за м².


In [2]:
!pip install -q rank_bm25
print("✓ rank_bm25 установлен")

✓ rank_bm25 установлен


In [3]:
with open("/content/rag_pipeline.py", "w", encoding="utf-8") as f:
    f.write('''import os
import re
import glob
from collections import Counter

from rank_bm25 import BM25Okapi

from config import (
    KNOWLEDGE_DIR, CHUNK_MAX_PARAGRAPHS, CHUNK_OVERLAP, TOP_K,
    HISTORY_MAX_MESSAGES,
)
from openai_client import get_embedding, get_embeddings_batch, chat
from chroma_store import (
    add_chunks, query_chunks, reset_collection, count, get_all_documents,
)

conversation_history = []

# ---------- Русские стоп-слова (короткий список) ----------
STOPWORDS = {
    "и", "в", "во", "не", "что", "он", "на", "я", "с", "со", "как",
    "а", "то", "все", "она", "так", "его", "но", "да", "ты", "к",
    "у", "же", "вы", "за", "бы", "по", "только", "ее", "мне", "было",
    "вот", "от", "меня", "еще", "нет", "о", "из", "ему", "теперь",
    "когда", "даже", "ну", "вдруг", "ли", "если", "уже", "или",
    "ни", "быть", "был", "него", "до", "вас", "нибудь", "опять",
    "уж", "вам", "ведь", "там", "потом", "себя", "ничего", "ей",
    "может", "они", "тут", "где", "есть", "надо", "ней", "для",
    "мы", "тебя", "их", "чем", "была", "сам", "чтоб", "без",
    "будто", "чего", "раз", "тоже", "себе", "под", "будет", "ж",
    "тогда", "кто", "этот", "того", "потому", "этого", "какой",
    "совсем", "ним", "здесь", "этом", "один", "почти", "мой",
    "тем", "чтобы", "нее", "сейчас", "были", "куда", "зачем",
    "всех", "никогда", "можно", "при", "наконец", "два", "об",
    "другой", "хоть", "после", "над", "больше", "тот", "через",
    "эти", "нас", "про", "всего", "них", "какая", "много",
    "разве", "три", "эту", "моя", "впрочем", "хорошо", "свою",
    "этой", "перед", "иногда", "лучше", "чуть", "том", "нельзя",
    "такой", "им", "более", "всегда", "конечно", "всю", "между",
    "это", "также", "какие", "это", "сколько", "что", "чем",
    "которые", "который", "которая", "которое", "которых",
    "ваш", "ваша", "ваше", "ваши", "наш", "наша", "наше", "наши",
    "этот", "эта", "это", "эти", "тот", "та", "те",
    "бы", "ли", "же", "уж", "вот",
}

TOKEN_RE = re.compile(r"[а-яёa-z0-9]+", re.IGNORECASE)


def tokenize(text: str) -> list[str]:
    """Простая токенизация с фильтрацией стоп-слов."""
    tokens = TOKEN_RE.findall(text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


# ---------- Чанкинг ----------

def smart_chunk(text, max_paragraphs=CHUNK_MAX_PARAGRAPHS, overlap=CHUNK_OVERLAP):
    paragraphs = [p.strip() for p in text.split("\\n\\n") if p.strip()]
    chunks = []
    i = 0
    while i < len(paragraphs):
        end = min(i + max_paragraphs, len(paragraphs))
        chunk = "\\n\\n".join(paragraphs[i:end])
        chunks.append(chunk)
        if end >= len(paragraphs):
            break
        i = end - overlap
    return chunks


def load_knowledge_files():
    files = []
    pattern = os.path.join(KNOWLEDGE_DIR, "*.md")
    for path in sorted(glob.glob(pattern)):
        with open(path, "r", encoding="utf-8") as fh:
            files.append((os.path.basename(path), fh.read()))
    return files


# ---------- BM25 ----------

_bm25_index = None
_bm25_docs = None
_bm25_metas = None
_bm25_ids = None


def build_bm25_index():
    """Строит BM25-индекс по тем же чанкам, что и в ChromaDB."""
    global _bm25_index, _bm25_docs, _bm25_metas, _bm25_ids

    ids, docs, metas = get_all_documents()
    if not docs:
        _bm25_index = None
        return 0

    tokenized = [tokenize(d) for d in docs]
    _bm25_index = BM25Okapi(tokenized)
    _bm25_docs = docs
    _bm25_metas = metas
    _bm25_ids = ids
    return len(docs)


def search_bm25(query: str, n_results: int = TOP_K) -> list[dict]:
    """Поиск по ключевым словам через BM25."""
    global _bm25_index
    if _bm25_index is None:
        build_bm25_index()
    if _bm25_index is None:
        return []

    tokens = tokenize(query)
    if not tokens:
        return []

    scores = _bm25_index.get_scores(tokens)
    ranked = sorted(
        enumerate(scores), key=lambda x: x[1], reverse=True
    )[:n_results]

    hits = []
    for idx, score in ranked:
        if score <= 0:
            continue
        meta = _bm25_metas[idx]
        hits.append({
            "text": _bm25_docs[idx],
            "source": meta.get("source", "unknown"),
            "chunk": meta.get("chunk", -1),
            "bm25_score": round(float(score), 4),
        })
    return hits


# ---------- Извлечение ключевых слов (без LLM) ----------

def extract_keywords(question: str, top_n: int = 6) -> list[str]:
    """Ключевые слова из вопроса — по частоте и длине.
    Без LLM, только регулярки и статистика."""
    tokens = tokenize(question)
    if not tokens:
        return []

    # Уникальные, сохраняем порядок первого появления
    seen = []
    for t in tokens:
        if t not in seen:
            seen.append(t)

    # Приоритезируем: сначала более длинные (обычно более информативные),
    # потом по частоте
    freq = Counter(tokens)
    seen.sort(key=lambda w: (-len(w), -freq[w]))
    return seen[:top_n]


# ---------- Индексация ----------

def build_index(verbose=True):
    reset_collection()
    files = load_knowledge_files()
    if not files:
        if verbose:
            print("[RAG] Папка knowledge_base пуста.")
        return 0

    ids, documents, metadatas, embeddings = [], [], [], []
    for filename, text in files:
        chunks = smart_chunk(text)
        for idx, chunk in enumerate(chunks):
            ids.append(f"{filename}::{idx}")
            documents.append(chunk)
            metadatas.append({"source": filename, "chunk": idx})

    BATCH = 64
    for start in range(0, len(documents), BATCH):
        batch = documents[start:start + BATCH]
        embeddings.extend(get_embeddings_batch(batch))

    add_chunks(ids, documents, embeddings, metadatas)

    # Пересобираем BM25-индекс поверх тех же чанков
    build_bm25_index()

    if verbose:
        print(f"[RAG] Проиндексировано чанков: {len(ids)}")
    return len(ids)


def ensure_index():
    if count() == 0:
        build_index()


# ---------- Embedding-поиск ----------

def search_embedding(query: str, n_results: int = TOP_K) -> list[dict]:
    query_emb = get_embedding(query)
    results = query_chunks(query_emb, n_results=n_results)
    docs = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    hits = []
    for doc, dist, meta in zip(docs, dists, metas):
        score = 1.0 - float(dist)
        hits.append({
            "text": doc,
            "source": meta.get("source", "unknown"),
            "chunk": meta.get("chunk", -1),
            "score": round(score, 4),
        })
    return hits


# ---------- RRF (Reciprocal Rank Fusion) ----------

def reciprocal_rank_fusion(*ranked_lists, k: int = 60, top_n: int = TOP_K):
    """Объединяет несколько ранжированных списков через RRF.
    Каждый элемент — словарь с 'text', 'source', 'chunk'."""
    scores = {}
    payload = {}

    for ranked in ranked_lists:
        for rank, item in enumerate(ranked, start=1):
            key = (item["source"], item["chunk"])
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
            if key not in payload:
                payload[key] = item

    ordered = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    result = []
    for key, rrf_score in ordered:
        item = dict(payload[key])
        item["rrf_score"] = round(rrf_score, 6)
        result.append(item)
    return result


# ---------- Генерация ----------

SYSTEM_PROMPT = (
    "Ты — ассистент компании RemStroy. Компания занимается ремонтом "
    "жилых квартир и установкой оконных и балконных блоков "
    "(2- и 3-слойных) по различным технологиям.\\n\\n"
    "ПРАВИЛА:\\n"
    "1. Отвечай ТОЛЬКО на последний заданный вопрос. "
    "Игнорируй предыдущие вопросы, если они не относятся к делу.\\n"
    "2. Не начинай ответ с фраз про отсутствие информации, "
    "если ответ на заданный вопрос есть в контексте.\\n"
    "3. Не упоминай темы, которых нет в текущем вопросе.\\n"
    "4. Если информации действительно нет — скажи об этом одной "
    "краткой фразой и остановись.\\n"
    "5. Отвечай на русском языке, по существу, без вводных слов."
)


def generate_answer(question, hits):
    context_parts = []
    for i, h in enumerate(hits, 1):
        context_parts.append(
            f"[Фрагмент {i} | источник: {h['source']}]\\n{h['text']}"
        )
    context = "\\n\\n---\\n\\n".join(context_parts)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Контекст:\\n{context}\\n\\nВопрос: {question}",
        },
    ]
    return chat(messages)


# ---------- Основной вход ----------

def rag_query(question: str) -> dict:
    ensure_index()

    # 1. Ключевые слова (без LLM)
    keywords = extract_keywords(question)

    # 2. Embedding-поиск
    embedding_hits = search_embedding(question, n_results=TOP_K)

    # 3. Keyword-поиск (BM25)
    keyword_hits = search_bm25(question, n_results=TOP_K)

    # 4. RRF — объединение
    final_hits = reciprocal_rank_fusion(
        embedding_hits, keyword_hits, k=60, top_n=TOP_K
    )

    # 5. Генерация по финальному набору
    answer = generate_answer(question, final_hits)

    conversation_history.append({"role": "user", "content": question})
    conversation_history.append({"role": "assistant", "content": answer})
    if len(conversation_history) > HISTORY_MAX_MESSAGES:
        conversation_history[:] = conversation_history[-HISTORY_MAX_MESSAGES:]

    return {
        "keywords": keywords,
        "embedding_chunks": embedding_hits,
        "keyword_chunks": keyword_hits,
        "final_chunks": final_hits,
        "chunks": final_hits,   # для обратной совместимости
        "answer": answer,
    }


def reset_history():
    conversation_history.clear()
''')

print("✓ rag_pipeline.py обновлён: гибридный поиск + RRF")

✓ rag_pipeline.py обновлён: гибридный поиск + RRF


In [4]:
import os, sys
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.chdir("/content")

# Сброс кэша модулей
for mod in ["config", "openai_client", "chroma_store", "rag_pipeline"]:
    if mod in sys.modules:
        del sys.modules[mod]

import rag_pipeline

# Пересобираем индекс (нужен для BM25)
n = rag_pipeline.build_index()
print(f"\n✓ Чанков: {n}\n")

# Тестовый запрос
result = rag_pipeline.rag_query("Сколько стоит установка трёхслойного окна на балконе?")

print("=" * 60)
print("КЛЮЧЕВЫЕ СЛОВА ИЗ ВОПРОСА (без LLM):")
print(" ", ", ".join(result["keywords"]))
print("=" * 60)

print("\n--- EMBEDDING-ПОИСК (топ-5) ---")
for i, c in enumerate(result["embedding_chunks"], 1):
    print(f"{i}. {c['source']} · score: {c['score']}")

print("\n--- BM25 (ключевые слова, топ-5) ---")
for i, c in enumerate(result["keyword_chunks"], 1):
    print(f"{i}. {c['source']} · BM25: {c['bm25_score']}")

print("\n--- ФИНАЛЬНЫЙ НАБОР ПОСЛЕ RRF (топ-5) ---")
for i, c in enumerate(result["final_chunks"], 1):
    print(f"{i}. {c['source']} · RRF: {c['rrf_score']}")

print("\n" + "=" * 60)
print("AI-ОТВЕТ:")
print("=" * 60)
print(result["answer"])

[RAG] Проиндексировано чанков: 30

✓ Чанков: 30

КЛЮЧЕВЫЕ СЛОВА ИЗ ВОПРОСА (без LLM):
  трёхслойного, установка, балконе, стоит, окна

--- EMBEDDING-ПОИСК (топ-5) ---
1. ceny_i_usloviya.md · score: 0.8351
2. ceny_i_usloviya.md · score: 0.8172
3. okna_pvh_3sloy.md · score: 0.7916
4. balkonnye_bloki.md · score: 0.7898
5. balkonnye_bloki.md · score: 0.7815

--- BM25 (ключевые слова, топ-5) ---
1. okna_pvh_3sloy.md · BM25: 2.6477
2. tehnologii_montazha.md · BM25: 2.1148
3. okna_pvh_2sloy.md · BM25: 1.9317
4. okna_pvh_2sloy.md · BM25: 1.8049
5. okna_pvh_2sloy.md · BM25: 0.2287

--- ФИНАЛЬНЫЙ НАБОР ПОСЛЕ RRF (топ-5) ---
1. okna_pvh_3sloy.md · RRF: 0.032266
2. ceny_i_usloviya.md · RRF: 0.016393
3. ceny_i_usloviya.md · RRF: 0.016129
4. tehnologii_montazha.md · RRF: 0.016129
5. okna_pvh_2sloy.md · RRF: 0.015873

AI-ОТВЕТ:
Трёхслойные ПВХ-окна стоят от 18


In [5]:
print("Длина ответа:", len(result["answer"]), "символов")
print("=" * 60)
print(result["answer"])
print("=" * 60)

Длина ответа: 32 символов
Трёхслойные ПВХ-окна стоят от 18


In [6]:
import rag_pipeline

# Тестовый запрос напрямую в LLM (без RAG)
from openai_client import chat

test_messages = [
    {"role": "system", "content": "Отвечай кратко на русском."},
    {"role": "user", "content": "Сколько стоит трёхслойное окно?"},
]
answer = chat(test_messages)
print("Сырой ответ LLM:")
print(repr(answer))
print("\nДлина:", len(answer))

Сырой ответ LLM:
'В среднем стандартное окно с трёхслойным стеклопакетом (двухкамерным) стоит **от 15 000 до 30 000 рублей** с установкой. \n\nБез монтажа — **от 10 000 до 20 000 рублей**. Точная цена зависит от размеров'

Длина: 200


In [7]:
import os, sys
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.chdir("/content")

# Сброс модулей
for mod in ["config", "openai_client", "chroma_store", "rag_pipeline"]:
    if mod in sys.modules:
        del sys.modules[mod]

import importlib
import openai_client
import rag_pipeline
importlib.reload(openai_client)
importlib.reload(rag_pipeline)

# === Тест 1: сырой ответ с finish_reason ===
from openai import OpenAI
from config import GEMINI_API_KEY, BASE_URL, CHAT_MODEL

client = OpenAI(api_key=GEMINI_API_KEY, base_url=BASE_URL)

# Соберём реальный контекст, как в RAG
question = "Сколько стоит трёхслойное окно?"
hits = rag_pipeline.search_embedding(question, n_results=5)
context_parts = [f"[Фрагмент {i} | источник: {h['source']}]\n{h['text']}" for i, h in enumerate(hits, 1)]
context = "\n\n---\n\n".join(context_parts)

messages = [
    {"role": "system", "content": rag_pipeline.SYSTEM_PROMPT},
    {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"},
]

print("=== Длина system-промпта:", len(rag_pipeline.SYSTEM_PROMPT), "символов")
print("=== Длина user-сообщения:", len(messages[1]["content"]), "символов")

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    temperature=0.3,
    max_tokens=500,
)

print("\n=== finish_reason:", response.choices[0].finish_reason)
print("=== usage:", response.usage)
print("\n=== Сырой ответ:")
print(repr(response.choices[0].message.content))
print("\n=== Длина:", len(response.choices[0].message.content or ""))

=== Длина system-промпта: 579 символов
=== Длина user-сообщения: 2219 символов

=== finish_reason: length
=== usage: CompletionUsage(completion_tokens=20, prompt_tokens=993, total_tokens=1489, completion_tokens_details=None, prompt_tokens_details=None)

=== Сырой ответ:
'Стоимость трёхслойного окна «под ключ» — от 18 000'

=== Длина: 50


In [8]:
# Упрощаем SYSTEM_PROMPT
with open("/content/rag_pipeline.py", "r", encoding="utf-8") as f:
    src = f.read()

# Заменяем промпт на короткий
import re
src = re.sub(
    r'SYSTEM_PROMPT = \(.*?\)\n',
    '''SYSTEM_PROMPT = (
    "Ты — ассистент компании RemStroy. Отвечай на русском языке "
    "по существу, опираясь ТОЛЬКО на предоставленный контекст. "
    "Если в контексте нет ответа — кратко скажи об этом."
)
''',
    src,
    count=1,
    flags=re.DOTALL,
)

with open("/content/rag_pipeline.py", "w", encoding="utf-8") as f:
    f.write(src)

print("✓ SYSTEM_PROMPT упрощён")

# Перезагружаем модуль и пробуем снова
import importlib, sys
for mod in ["rag_pipeline"]:
    if mod in sys.modules:
        del sys.modules[mod]
import rag_pipeline
importlib.reload(rag_pipeline)

result = rag_pipeline.rag_query("Сколько стоит трёхслойное окно?")
print("\nДлина ответа:", len(result["answer"]))
print("Ответ:")
print(result["answer"])

✓ SYSTEM_PROMPT упрощён

Длина ответа: 65
Ответ:
Стоимость трёхслойного ПВХ-окна «под ключ» составляет **от 18 000


In [9]:
with open("/content/openai_client.py", "w", encoding="utf-8") as f:
    f.write('''from openai import OpenAI
from config import (
    GEMINI_API_KEY, BASE_URL, CHAT_MODEL, EMBEDDING_MODEL,
    TEMPERATURE, MAX_TOKENS,
)

_client = OpenAI(api_key=GEMINI_API_KEY, base_url=BASE_URL)


def get_embedding(text):
    response = _client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


def get_embeddings_batch(texts):
    response = _client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


def chat(messages, max_tokens=2048):
    """Отправляет сообщения в Gemini. max_tokens повышен до 2048,
    потому что Gemini через OpenAI-интерфейс часто игнорирует
    меньшие значения и обрывает ответ на 20 токенах."""
    response = _client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=max_tokens,
    )
    choice = response.choices[0]
    print(f"[DIAG] finish_reason={choice.finish_reason}, "
          f"completion_tokens={response.usage.completion_tokens}")
    return choice.message.content
''')

print("✓ openai_client.py обновлён: max_tokens=2048 + диагностика")

✓ openai_client.py обновлён: max_tokens=2048 + диагностика


In [10]:
import sys
for mod in ["config", "openai_client", "chroma_store", "rag_pipeline"]:
    if mod in sys.modules:
        del sys.modules[mod]

import rag_pipeline

result = rag_pipeline.rag_query("Сколько стоит трёхслойное окно?")
print("\n=== ПОЛНЫЙ ОТВЕТ ===")
print(result["answer"])
print("\n=== Длина:", len(result["answer"]))

[DIAG] finish_reason=stop, completion_tokens=90

=== ПОЛНЫЙ ОТВЕТ ===
Стоимость трёхслойного ПВХ-окна «под ключ» составляет **от 18 000 ₽ за м²**.

Итоговая цена зависит от площади, профиля, стеклопакета и фурнитуры.

Также действуют скидки:
* При заказе от 5 окон — скидка 7%;
* При заказе от 10 окон — скидка 12%.

=== Длина: 245


In [11]:
import subprocess
import time
import os

os.chdir("/content")

# Убиваем старые процессы, если остались с прошлой сессии
!pkill -f "python main.py" 2>/dev/null
!pkill -f cloudflared 2>/dev/null
time.sleep(1)

# Запускаем сервер
server = subprocess.Popen(
    ["python", "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(3)
print("Сервер запущен (PID:", server.pid, ")")

# Запускаем cloudflared
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(8)
print("Туннель запущен (PID:", tunnel.pid, ")")

# Ищем URL
found_url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    if line and "trycloudflare.com" in line and "https://" in line:
        for w in line.split():
            if w.startswith("https://") and "trycloudflare.com" in w:
                found_url = w
                break
        if found_url:
            break

if found_url:
    print("\n" + "=" * 60)
    print("ПУБЛИЧНАЯ ССЫЛКА:", found_url)
    print("=" * 60)
else:
    print("\n⚠ Ссылка не найдена. Проверьте вывод туннеля.")

^C
^C
Сервер запущен (PID: 24553 )


FileNotFoundError: [Errno 2] No such file or directory: '/content/cloudflared'

In [12]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print("✓ cloudflared установлен")

✓ cloudflared установлен


In [13]:
import subprocess
import time
import os

os.chdir("/content")

!pkill -f "python main.py" 2>/dev/null
!pkill -f cloudflared 2>/dev/null
time.sleep(1)

server = subprocess.Popen(
    ["python", "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(3)
print("Сервер запущен (PID:", server.pid, ")")

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(8)
print("Туннель запущен (PID:", tunnel.pid, ")")

found_url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    if line and "trycloudflare.com" in line and "https://" in line:
        for w in line.split():
            if w.startswith("https://") and "trycloudflare.com" in w:
                found_url = w
                break
        if found_url:
            break

if found_url:
    print("\n" + "=" * 60)
    print("ПУБЛИЧНАЯ ССЫЛКА:", found_url)
    print("=" * 60)
else:
    print("\n⚠ Ссылка не найдена. Проверьте вывод туннеля.")

^C
^C
Сервер запущен (PID: 26029 )
Туннель запущен (PID: 26044 )

ПУБЛИЧНАЯ ССЫЛКА: https://contract-sacred-florence-factors.trycloudflare.com
